# Stage KiTS23 v0.1.4 on Kaggle — shard 2 of 3 (CPU, no GPU quota)

Downloads about 14 GB of the pinned KiTS23 files straight from HuggingFace into
this notebook's output, checking every file's size and SHA-256 against the
project manifest. Kaggle keeps at most about 20 GB of output, so the data is
split over 3 notebooks.

*Settings*: **Accelerator = None**, **Internet = On**, keep the notebook **Private**
(the data license is CC BY-NC-SA 4.0). Then **Save Version → Save & Run All**.
Notebooks 01 and 03 take the outputs of all 3 shards as input.

Generated by `scripts/data/build_stage_notebook.py`; edit the repository, not this notebook.

In [ ]:
import os, sys, subprocess
for sub in ("scripts/data", "data/sources"):
    os.makedirs(f"/kaggle/working/project23d/{sub}", exist_ok=True)

Write the project code (generated from the repository).

In [ ]:
%%writefile /kaggle/working/project23d/scripts/data/stage_kits23.py
"""Download one shard of KiTS23 v0.1.4 from the pinned URLs and verify it (D-015).

Kaggle saves at most about 20 GB of notebook output, and the images are 42 GB, so
the 489 images are split into ``--n-shards`` contiguous shards of similar size.
Shard 1 also holds all 489 segmentations (0.14 GB). Each file's size and SHA-256
are checked against the manifest while it streams; a bad file is retried, then
reported. Example::

    python scripts/data/stage_kits23.py --manifest data/sources/kits23_v0.1.4_files.tsv \
        --shard 1 --out-dir /kaggle/working/kits23

Writes ``images/``, ``segmentations/`` (shard 1 only) and ``stage_record.json``.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import sys
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

FOLDER = {"image": "images", "segmentation": "segmentations"}


def load_rows(tsv_path: Path) -> list[dict]:
    with open(tsv_path, newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh, delimiter="\t"))
    for r in rows:
        r["size_bytes"] = int(r["size_bytes"])
    return sorted(rows, key=lambda r: (r["kind"], r["case_id"]))


def plan_shard(rows: list[dict], shard: int, n_shards: int) -> list[dict]:
    """Rows for ``shard`` (1-based): a contiguous run of images by cumulative size."""
    if not 1 <= shard <= n_shards:
        raise ValueError(f"shard must be in 1..{n_shards}, got {shard}")
    images = [r for r in rows if r["kind"] == "image"]
    total = sum(r["size_bytes"] for r in images)
    picked, done = [], 0
    for r in images:
        # a file belongs to the shard in which its midpoint falls
        if int((done + r["size_bytes"] / 2) * n_shards // total) + 1 == shard:
            picked.append(r)
        done += r["size_bytes"]
    if shard == 1:
        picked += [r for r in rows if r["kind"] == "segmentation"]
    return picked


def fetch(row: dict, out_dir: Path, retries: int = 3) -> dict:
    dst = out_dir / FOLDER[row["kind"]] / f"{row['case_id']}.nii.gz"
    status = "MISSING"
    for attempt in range(1, retries + 1):
        if dst.exists() and dst.stat().st_size == row["size_bytes"]:
            h = hashlib.sha256()
            with open(dst, "rb") as fh:
                while chunk := fh.read(8 * 1024 * 1024):
                    h.update(chunk)
            if h.hexdigest() == row["sha256"]:
                return {"case_id": row["case_id"], "kind": row["kind"], "status": "OK", "attempts": attempt - 1}
        part = dst.with_suffix(".part")
        try:
            h, size = hashlib.sha256(), 0
            with urllib.request.urlopen(row["url"], timeout=120) as resp, open(part, "wb") as fh:
                while chunk := resp.read(8 * 1024 * 1024):
                    h.update(chunk)
                    size += len(chunk)
                    fh.write(chunk)
            if size != row["size_bytes"]:
                status = "SIZE_MISMATCH"
            elif h.hexdigest() != row["sha256"]:
                status = "HASH_MISMATCH"
            else:
                part.replace(dst)
                return {"case_id": row["case_id"], "kind": row["kind"], "status": "OK", "attempts": attempt}
        except OSError as e:
            status = f"ERROR {type(e).__name__}: {e}"
        part.unlink(missing_ok=True)
        time.sleep(2 * attempt)
    return {"case_id": row["case_id"], "kind": row["kind"], "status": status, "attempts": retries}


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--manifest", type=Path, required=True)
    ap.add_argument("--shard", type=int, required=True)
    ap.add_argument("--n-shards", type=int, default=3)
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--workers", type=int, default=8)
    args = ap.parse_args(argv)

    t0 = time.time()
    rows = plan_shard(load_rows(args.manifest), args.shard, args.n_shards)
    for kind in {r["kind"] for r in rows}:
        (args.out_dir / FOLDER[kind]).mkdir(parents=True, exist_ok=True)
    gb = sum(r["size_bytes"] for r in rows) / 1e9
    print(f"shard {args.shard}/{args.n_shards}: {len(rows)} files, {gb:.1f} GB", flush=True)
    results = []
    with ThreadPoolExecutor(args.workers) as pool:
        for i, res in enumerate(pool.map(lambda r: fetch(r, args.out_dir), rows), 1):
            results.append(res)
            if res["status"] != "OK" or i % 25 == 0 or i == len(rows):
                print(f"[{i}/{len(rows)}] {res['case_id']} {res['kind']}: {res['status']}"
                      f"  ({time.time() - t0:.0f}s)", flush=True)
    failed = [r for r in results if r["status"] != "OK"]
    record = {"shard": args.shard, "n_shards": args.n_shards, "files": len(rows),
              "images": sum(r["kind"] == "image" for r in rows),
              "segmentations": sum(r["kind"] == "segmentation" for r in rows),
              "gb": round(gb, 2), "ok": len(results) - len(failed), "failed": failed,
              "minutes": round((time.time() - t0) / 60, 1)}
    (args.out_dir / "stage_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in record.items() if k != "failed"}, indent=1))
    if failed:
        print(f"STOP: {len(failed)} files failed; rerun this notebook")
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /kaggle/working/project23d/data/sources/kits23_v0.1.4_files.tsv
case_id	kind	size_bytes	sha256	url
case_00000	image	225959569	cdae5f3e0fbc7c98ab0430b3de42677abda2c1cf93ae0f86bd29fb8606688cb7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00000.nii.gz
case_00000	segmentation	776578	93b36b187639981d1473c30b60eb134c69ed207bbbfe6564225830db2d7e5579	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00000.nii.gz
case_00001	image	276387358	3b861149f13a1e8a0b7cae23f624e6c335f538ee9fb6ecc0625b2f062879bed6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00001.nii.gz
case_00001	segmentation	851533	a17e5ecdc4abc44a90ee277f9a61b932c90ee008b82c33fe407514a135f3edbb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00001.nii.gz
case_00002	image	101967812	58fcd6c4e59c747c2927b4ff12305f553abd33454cff6533082706b946173201	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00002.nii.gz
case_00002	segmentation	373942	3966f0fdf015d396a5296d18b4ffbbb91f6c946035a352af581b36664800bf4d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00002.nii.gz
case_00003	image	118681596	63d267967cd02889415de2978903df36703cd7b5181e1226dc91d4832f7d5586	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00003.nii.gz
case_00003	segmentation	379266	64e24f4cd3e90e57228a2dd4598244a208f58034f53f93f07019e99124368440	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00003.nii.gz
case_00004	image	25269467	a9af51c0435086ac7007570c8bc66ceb2a54ecfb7ec530639150635d326f8f1f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00004.nii.gz
case_00004	segmentation	101172	a87e1e94009509dc29b85c5625dec53a153529f5cfc4eec5cd4a0df446f75be9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00004.nii.gz
case_00005	image	313477305	986d4186183ceefba226cf667f6c07b010c60f6b4ea00905a60b805d05300279	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00005.nii.gz
case_00005	segmentation	1039494	6e04feeb30a0741b675f8dbb547f4c96b13c2aaf2e1463ee2d5b90cb76dab48d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00005.nii.gz
case_00006	image	78010682	884cc817625dd041615a83ac378240675d599a42a1e0dd86e6fe88bd17f1c174	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00006.nii.gz
case_00006	segmentation	250640	f39f759a09802f890eeb7cf9aca4aecb7086d5a60121f4880ab10cc220621b99	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00006.nii.gz
case_00007	image	25826334	1261e949b46d05b0da75ac7fa73f0c6a51aa13a607d95d89fefc232b6b3ca20b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00007.nii.gz
case_00007	segmentation	117467	17fb280af53524c1d547c5e3f6bfd28fcc533f4cdb83d54e3612516ea3295e86	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00007.nii.gz
case_00008	image	107068349	ae2719417e36da5689aade422a0b94bd896c4b33d7897ff3ab7e30129ae119c6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00008.nii.gz
case_00008	segmentation	381662	7ac4f084ac03462bbfc9ab0d1e8936cfb9a32a95f4a1bdea1425760271d8d891	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00008.nii.gz
case_00009	image	34134599	492de10723f687b21b589ba1660615cac5eb4a8a785580a807be4eddca7c1756	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00009.nii.gz
case_00009	segmentation	126558	d854a3dc9568ba0ab9b0da23d8251beda0e2c5f722777f587dabb2ff2cd4e5aa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00009.nii.gz
case_00010	image	23541391	8e30ba81aa0422e2091b749b7c30989aeb102e01afcec826045162b437ed0ba4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00010.nii.gz
case_00010	segmentation	99339	9886b2c9ec5dfa57a7564c14d7a856425be503474e8e6ce8a09202fe5c3e472e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00010.nii.gz
case_00011	image	41268802	f2d346935837ad763840155ee07fde34a6698e03d207db360321a491969d4997	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00011.nii.gz
case_00011	segmentation	136203	1d7fd2eec44517c521e634c2dd1c2c68524c0eb732cf1e2187c98428dcb8420d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00011.nii.gz
case_00012	image	43185427	039c97ac928af4dbb4f959cbaf2c7403e1c9590321ffdb3f2d86a355261cd846	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00012.nii.gz
case_00012	segmentation	143415	09636356304f22b7fe92de905bff0b8f289c56eaa064c6c9766d27918924cbff	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00012.nii.gz
case_00013	image	44485657	77e7eef8f9935192f526246f7ba5de0f3ef569ab1edc3ff18e930db8968ed7f5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00013.nii.gz
case_00013	segmentation	149530	e24264b25c7c832f1be09d3ee65ff867efedf3ee4dbd354353862cf3d20806e6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00013.nii.gz
case_00014	image	192486296	80ae025ada8b7f50fa9cbb223c81bb8770d00dc02db5d49b7f075b760380b664	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00014.nii.gz
case_00014	segmentation	561766	64f6578a3ea3683728bc67f7f0072e87a4aafa6f9cd94df1773482e1b2af51d6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00014.nii.gz
case_00015	image	23795573	b6b216621db1c03a441b81f32540467767270f89e8b500e2012ccaa75bb12869	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00015.nii.gz
case_00015	segmentation	146162	2cd1f5998ea641398987fb5f6b86d7cbb6fa8a769ff3b852d187dc68eca77498	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00015.nii.gz
case_00016	image	86929344	9f741f28473b79ed7e53c5dcc5bc7d8872a6d0449dbb829a903eb05ae4f8e820	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00016.nii.gz
case_00016	segmentation	285636	a4aa2ffee821e2e6c9c61f93d4a8f0ea55f7c81b1d2b61fff62e01f26721932c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00016.nii.gz
case_00017	image	49940149	fb7c9074168a4a7a258f8a300b4704af42e23a63e55027834fe1e44e55b473f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00017.nii.gz
case_00017	segmentation	155475	18ca85dfca7d4c47b5e1b7c63ad21ef997a25ff9760a9e58953564784ebe9866	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00017.nii.gz
case_00018	image	49984866	4a55b9904bb5e3bb4738721324f49b0bb4bf4e813fee7a635cd124db5f9c40d1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00018.nii.gz
case_00018	segmentation	204149	7c0e48177cae58f207b6c3dae17b033cb38bf13e49c6a641cca6f3dea5d62954	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00018.nii.gz
case_00019	image	51644458	0b2cd96ac11bebc5f07e071398e273d418e13476d43e1229927c4c33fa5e03be	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00019.nii.gz
case_00019	segmentation	184913	0fcbaaf6c96ac0c99ddcf8d57c1b871c0bb3697e315c786d00b81b21ebb6a430	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00019.nii.gz
case_00020	image	43148211	0a9f9f55949dd5fcd59ca49c27662d9956d0bb87cbdab50c7f5ed347bd9e732c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00020.nii.gz
case_00020	segmentation	142354	6f8c99e7682ceab76133513424867bfc642b2264bda34768ad997f3613f4d79a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00020.nii.gz
case_00021	image	17267757	97562e5d35abe55192fbbaacd04f05171343ab440c5ff592c0fad729d001a9f7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00021.nii.gz
case_00021	segmentation	73888	0292f2683b376043f478788624a8fdd924ce736c98626378fb897d4723991b50	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00021.nii.gz
case_00022	image	235518269	365cb0082c4073eb7e60f7c759e8f6d7feef6a168f4f55edc2324b243518a452	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00022.nii.gz
case_00022	segmentation	747112	ca66ecc6587ff33d0b1c2795cdd4257847f856a13caa25b67e733f371aa67868	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00022.nii.gz
case_00023	image	45338328	b5a948824220f88ae3515234110034af1f2ca07a4b00bcaa597bd06d33b5b388	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00023.nii.gz
case_00023	segmentation	184343	dc9556a4cd6dff0dc430b59961dc13c7acf05e61132ddff80cc341482ba19695	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00023.nii.gz
case_00024	image	43319114	b90540df3dd4ab0ec4cad772504d1be9471229c01a14f6cd7f0e9fb3a4523e71	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00024.nii.gz
case_00024	segmentation	136773	1506e5ddbc7342f3f352f7861159f85501182c804b96082d90c7149bf1d464a8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00024.nii.gz
case_00025	image	42608517	0b93a34f9ab521321eb7990920311912305131803d3058814f1d4dc16045a04c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00025.nii.gz
case_00025	segmentation	232757	e0bfd60bf7d457391e1056371e9793ef3a1c6bf1e1eec67defb710da70b77222	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00025.nii.gz
case_00026	image	128953647	effb38cede495d9c38e46e496117c4ebb50f912223d077f45cacb9eb0aa1a9da	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00026.nii.gz
case_00026	segmentation	455099	cd0604b932d15c9224250fbb46c1543c8ee7dc08bf97086d802bb84b016f923e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00026.nii.gz
case_00027	image	300960665	7432ae26ab57ab2bebdce01bc420f79edac2ad6728351128bd59736816ee58e8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00027.nii.gz
case_00027	segmentation	922854	40626415d68849069702f1dec732c7d8299b7ed1af672c27da316f785b6911db	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00027.nii.gz
case_00028	image	45001658	b41fdc5b7a7bb0a53a3ac25bbb875a8fe437a96622f6839201f8909af66612ce	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00028.nii.gz
case_00028	segmentation	177466	96f414d4561ee1eff1873363259c7b6e0952c3508a05193cdafb25a6ed090422	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00028.nii.gz
case_00029	image	62472569	657fdb4c2ef5a66e4580e64bdacbaf9a0e9034cb3fd390b28337081cd3142e45	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00029.nii.gz
case_00029	segmentation	186212	a8bc0bf499a2e74906d9c80fbb13a7bbacf0f6d42ea22ed2802df80c57cb2b62	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00029.nii.gz
case_00030	image	18676453	db5d5dd393c0fcbcdfd0d41e242c1c46138d6308765992d5b79e9a41f01b4ee2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00030.nii.gz
case_00030	segmentation	124464	c794f6e9e404f19b61dc9042a6c3b389c3b7bddfa84d4d40fcf118f42ff0b4d1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00030.nii.gz
case_00031	image	50504430	8e4baf209f156e6319cdbd61efdcb8bb518a9e863d86cb67ef43fe53cd3cc156	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00031.nii.gz
case_00031	segmentation	198735	6441070d4624f8ddbeae70f5aed2a0c1eabe4b57b1c9a43f684c94b2c2d15997	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00031.nii.gz
case_00032	image	75286311	00f43320501a2bc31b13176add47ae81793f9097df47c750c4ac4af9910f730d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00032.nii.gz
case_00032	segmentation	259729	7d5e320367dfd744c6819b797ed036210b1002e7db79247d207b19b1aa8996c8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00032.nii.gz
case_00033	image	191873285	94524a641d762fd1692811504f6276a970f1738de4641683453b0b5fd346a3f2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00033.nii.gz
case_00033	segmentation	589120	3c08ba43fa86869c5adc7df8e141b0161b6dc07c79472f3b65181379843b19be	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00033.nii.gz
case_00034	image	52752885	55213482fc646bf30a736fc1e8620a0925b30030cbfbbd6488d47bd059f313ac	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00034.nii.gz
case_00034	segmentation	174741	746bdf10766358ea6c9fc42c5b24e4c0589fdca86386cb762f5e080d57ae4814	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00034.nii.gz
case_00035	image	39633847	0fab3e5adaf5f525573875ed540d24f351e3cb8c23004739c1a653bfa5b66a0c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00035.nii.gz
case_00035	segmentation	145119	023402f77c4f0e922ac24840d80b69d4dbf8e6de10c010252396f87169013c97	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00035.nii.gz
case_00036	image	78614692	e3a61271d58bbb2ebe5c9018ea7246bad81350bcc4bab0c7ca758f8ffe6822af	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00036.nii.gz
case_00036	segmentation	267230	e6f2537fad1d42bb7346484d3e85ed0ea162c370dfbd61a816ba870d6f4db58b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00036.nii.gz
case_00037	image	40424057	31b95c03f5df0527f0943db956904a70e6c9cd779df3c6f9755858abeab4f64d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00037.nii.gz
case_00037	segmentation	169966	affb7d353494206b550dd1cb7a13a89d2faa2479514cb07f2357204e571fbd24	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00037.nii.gz
case_00038	image	15415661	0050172529386ea12ff21b2871e2afa0bc917f8ec2b8fc0fc5e9c0d2e298d6a3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00038.nii.gz
case_00038	segmentation	69058	59c64e5645aa82b06ca725dc7f987658caa5d677953548797bbb3f4192641e85	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00038.nii.gz
case_00039	image	44953099	ba31bba727acdff50aa43244d17872c75cdf69ad6dc493e4a52c36c44e953e3c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00039.nii.gz
case_00039	segmentation	137515	23ff24cc9d8f425a5c46db6735a3d919b1c907bff9559c129a3713eb830ebb36	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00039.nii.gz
case_00040	image	81954860	d0098068040f8fac079925d489428ef3b486d63975f51778c1dd981d5e7ff17f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00040.nii.gz
case_00040	segmentation	308147	aac455071161bb89738b2a3606380100389f0c746b7fda44b9a1b8f6012c3311	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00040.nii.gz
case_00041	image	25224078	bebf5e542eaa5731c71717b7654065fa4313c551c1cdf57f7cea426ceff028c2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00041.nii.gz
case_00041	segmentation	104609	681189db32e2874770f3afaf23364a3bdffeecd69c811c1927253f36b072e98b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00041.nii.gz
case_00042	image	136618407	07c3503b756e9290261afd92f4b51b141c3559ff6ea42a670169ff726a713f0d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00042.nii.gz
case_00042	segmentation	556283	1e8cd9aa129dc6448ff36ffd8dc51e325c38660dacd26f8ce43811b7ed489df4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00042.nii.gz
case_00043	image	70644303	e3780331e973be89132aedc51c51995e8ca729391d52e194f25f7cec08a659a5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00043.nii.gz
case_00043	segmentation	249205	bdd90320e1c6ab929e4c2c68820c771a512d0360bcc5a7bc36a7b19f1677de19	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00043.nii.gz
case_00044	image	38419186	4d7f8fd27cb08c8b935663757f864b3bb4113eb4af5b8ecdc9dda41ab48cdf3d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00044.nii.gz
case_00044	segmentation	156729	8e79ea88a9a0abd0c271b6826ab917431c20d45d4da907eae83b46eaef7449bf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00044.nii.gz
case_00045	image	25584430	ad4ddba4d11c3272dac7a35e2d4414a872b215fad2c85be7af64d7bd0caa80c8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00045.nii.gz
case_00045	segmentation	112484	5964fef7a14e174d1511bac6cab446449e51be9245c706977dd6c34f7f596ac3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00045.nii.gz
case_00046	image	59261129	4f83900ab55be64e53d2b88fab5657e5d92fc10ecf30a48770615e8d2b4e0a62	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00046.nii.gz
case_00046	segmentation	227414	3416bcbc18e1ee3915fbbfd9ff5b597691ac45551ff34b4be9f4042bfcd7b449	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00046.nii.gz
case_00047	image	63514124	83322cd75cb52290e64edd99dc7e23c616b8e31c62fe238f5f16f45bea6d477a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00047.nii.gz
case_00047	segmentation	206080	5f3920bb54485ce98b301e5cb3226bbaceba5179214f266c9a63d8efa2dd2c8b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00047.nii.gz
case_00048	image	30867959	a4762f5c8cbd30ab74cf4b16b4806a73b0f1234efb247d068a9b1c388d5ba8c4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00048.nii.gz
case_00048	segmentation	117384	5f956fdd990fce7f1945e252aa5e9a3e538e5ce26a2f687a436094d47ed20cd5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00048.nii.gz
case_00049	image	289194549	d0d449e4ca322aab438a34cebe0eb47b0ea68557593ef58f1912dfeac3c5fd32	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00049.nii.gz
case_00049	segmentation	900332	bd779260e3d6b8780567e8f3cbd96b05af75361d863621461cf379ab4dad858d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00049.nii.gz
case_00050	image	40193355	ee2983b3af2e20f2e130279b9415183c99d0d4b813aead6569a905b0197ca40b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00050.nii.gz
case_00050	segmentation	182397	efbea12ff7c017136d085a7fdd7b265a36f2ff4d9a5663b58c87fd9bcfe553f6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00050.nii.gz
case_00051	image	25645257	e52fc89a7156d19cfefcb7f8794c01e1673a5c77218361a3723d0886f53cb3fb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00051.nii.gz
case_00051	segmentation	121079	2a437f96b3f0f1dd500b082db4d53bb333e9e3a99cd6cc449fea7ebeb580f44d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00051.nii.gz
case_00052	image	256991563	e4f32880944933ce2f733cc75bc1a5284b0a846ed77161e995fd3d277aaec88a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00052.nii.gz
case_00052	segmentation	865567	66bfbcf51b731ddd808aa306a310d8c62340a611f8298348d9e3a47eb600831a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00052.nii.gz
case_00053	image	251298306	ed3821653b378580bee47a0b975ba8a3d659b95408a6010c8159fa9ecd6ab23c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00053.nii.gz
case_00053	segmentation	756498	3dc371962c4bb4c68629a56db573526d9ee7286ec27ce1d55ba9e2e6d5b98fba	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00053.nii.gz
case_00054	image	41865608	2e1670b9a7d9378c1e7d68242a08383d00c3c0755b74d5b7a46552656f4864c0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00054.nii.gz
case_00054	segmentation	154789	e6e149a54d40a9c43883d6cffb20528d3771f03b6c854912b8ad6d9d78a178c8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00054.nii.gz
case_00055	image	43342417	bdee52e4c1b3b789142b167b77257e394d8db579e26724488dd4dc1a2dbf0959	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00055.nii.gz
case_00055	segmentation	176139	db04ccffa7a33d6fd3567e4bacc7b034573359835e461a42fb6f53eb41723980	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00055.nii.gz
case_00056	image	45668159	bceeed3355ab099678d5ef009cb8a0258eac51fb6fc2530017fe702434c46140	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00056.nii.gz
case_00056	segmentation	179988	b6f46c33a82ea097e0bc06fe53320419c189a3fcf8bed341f90d60ead6ac7d14	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00056.nii.gz
case_00057	image	37609735	e7f73fded2c62e41d3b835bd8d65c075dd0b930347e16c5a55b854c6493ebdb6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00057.nii.gz
case_00057	segmentation	144109	4b9b83fa1989f6dd880c2c5718101f53448d08d0a5e589d417a49b417c1e9832	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00057.nii.gz
case_00058	image	43100380	6909f6f33d6ad42d48a04b2a392af6ff05431b5021a4aac0d9a00c1dde80530b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00058.nii.gz
case_00058	segmentation	169907	e2d30a6c065a132555bb71c6b9f920eeecc56114a76e82476952a50123edf493	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00058.nii.gz
case_00059	image	324407591	67a94a63adcee23ee66bba448d3229c47a1d33029524dde26dfa7d8fa1c2afae	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00059.nii.gz
case_00059	segmentation	944873	b00c591185c7a363ca6266f65e2d27ff4766fca58e9db5be5825d55d043ef06d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00059.nii.gz
case_00060	image	62615177	f4a8660114ba14a05f354d37511b50a0e6abfc26d6e0409b7b1bf6c2a8147fdd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00060.nii.gz
case_00060	segmentation	214556	bb9ab11021ce56f80c19294cd2f7e3e1c88c8430cbb8ce8c56665a269a64d604	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00060.nii.gz
case_00061	image	11577207	95eb42261e22f552e2b6bda39c2d72e3fbc376983d6bf9485122e968609f2cbf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00061.nii.gz
case_00061	segmentation	61404	abdfdc06322c4e04d00f71549dd025154ae0fad11fb77dd1656f014492c2fc3e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00061.nii.gz
case_00062	image	32540525	abaac9f1ca45105517719b9bbedf402214ebf12050305c70904a982ae59b084a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00062.nii.gz
case_00062	segmentation	117988	dc6b65205899cca9ae2f2b320c7ecf2d77ec052ba4a7e2390ba1308d7bdc6802	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00062.nii.gz
case_00063	image	231715658	0e27c6255a9305feb5bc4e95f48eb36c10c4d35a89acc0f3838d0629f314304c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00063.nii.gz
case_00063	segmentation	750661	9b556629c169d3837900b4d5512b201dd190e7769ff104f2cd62d20aa989ede2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00063.nii.gz
case_00064	image	21721710	d8de0bdfa099c9cd4fda5a69ed188f57cf01849e676d31473f5f0df4a83c832d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00064.nii.gz
case_00064	segmentation	87768	e61603893473b987be2c3be171c520bad145545071a1b87231978ba53576960d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00064.nii.gz
case_00065	image	48274302	6ff1b71bd873306c88e90fa463cbd740bbeb79cb5c3bc8b5089e3fff81622e6e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00065.nii.gz
case_00065	segmentation	193659	905033c60d3c647b625f06f542104070e408614760cf064b4bbc7f83780072e1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00065.nii.gz
case_00066	image	186603335	d8fbf08d1628afa013c3b0050b6f7b60ff5e184c295bea2ee5b0d173459ed82f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00066.nii.gz
case_00066	segmentation	597992	04fc1ff9bd600ae42f72f1a3cf46432edb2e6cdb03a2c8950d0513864c8f9d66	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00066.nii.gz
case_00067	image	134477869	a819c539f52b093658cf59afcb08f7f66b41adff9fec05125c47c2f5186b1f95	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00067.nii.gz
case_00067	segmentation	514360	7f945831b40d5a2b49f82d8fda89c4c8273a25ee65b7af2f919499a843843c4f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00067.nii.gz
case_00068	image	300829627	274e408499a644791a18e8287b10b8124932d28ad5ae6f80f3fd9a6cbf94d48c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00068.nii.gz
case_00068	segmentation	940872	11546be7fa5f2c8442f6470fd4ce5a7974959b9457ecfad23b90b536f12edf79	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00068.nii.gz
case_00069	image	46183471	f56e721684cdfb9d43e7860dd002e70858a33b0549bafe8c21be987c193c3592	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00069.nii.gz
case_00069	segmentation	167424	7170188517e02aab9627c2bd8a133c0f4399b20f878f174e39a6080f9aa9dae5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00069.nii.gz
case_00070	image	29478344	18025b039e3a098437eac420ef50d2fa3cb1ac879a78ae339037d90ef5330126	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00070.nii.gz
case_00070	segmentation	130320	3fd031a401cefbc6fb3410dbdff581281c6676f4c7c75095cfd041529cecc5df	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00070.nii.gz
case_00071	image	268825824	f7caca0580f003bc43e6d2d795d37e252665a31c949c2486fb55d835b675b25a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00071.nii.gz
case_00071	segmentation	801270	2a8716c22a7b0ef6dcd91d1fa9f04de1949aa5ed19aff85ab6aceefa6f973034	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00071.nii.gz
case_00072	image	79384535	2be7605792b5cc9c3303d1ccd4547da8b5839f75a3296ffc8826668de180cc56	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00072.nii.gz
case_00072	segmentation	269927	2390a58a531c01363b6a52ead1b2e01be4d84e375690dcf4eba012ed810a6c82	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00072.nii.gz
case_00073	image	66227643	54da755f9b760abea084d3cbe31a6279c3fc254a68f99733b2df19709c7b22f5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00073.nii.gz
case_00073	segmentation	228881	8cf0ee7326a9ef6099f8a6f78e7dfb10edddb108d22fcf7b8451317a4395f581	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00073.nii.gz
case_00074	image	39078517	1eb8693c31ac80dd360a3f91b610a60136ad276a5b334545c3f8e64765de5f17	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00074.nii.gz
case_00074	segmentation	134800	db0fd6ba61d2b1c9979798ff1390e3ff2bd79e8ea4bc2c0b8da328d125c6dbff	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00074.nii.gz
case_00075	image	44705075	44f3ac4045ce34305eadfc48af7f11782c7699ed986d71089ec82f744cdbb64e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00075.nii.gz
case_00075	segmentation	135615	3ad19ea44676ce3b5d000fc29d3c68cb40abf1eac66ef740f6b1261c79c30181	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00075.nii.gz
case_00076	image	32585283	24e7268fcd904b5d92aeddc37d6e2714ffb080de7c8e8497482c64116ce2f3e6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00076.nii.gz
case_00076	segmentation	107180	8fec8c2fb3c17fbe5f34fa44df93c620e9088730a89d465c295030e4919d61f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00076.nii.gz
case_00077	image	42628319	64d7f1fb5be850c6b30051e5244a7cfdfcdea1e185f8f8bea7e6056f8f1bd0c9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00077.nii.gz
case_00077	segmentation	136582	9f88229187d4c5bd13ec3ac65dcc2deaf6827932ac275fa7c80246cb351b6bac	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00077.nii.gz
case_00078	image	132219049	08563035343423b6011eddfe64e16447b2e9b8a62fbfffc0be97103a10664e44	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00078.nii.gz
case_00078	segmentation	536428	9a45390a88694dc9771695df24931ab43f6cf25088f9aabe495c06a777dd3048	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00078.nii.gz
case_00079	image	60760713	2062d34914ce3c0d00d7a92320a2cbc8e8f82d88217feeff4df2d6c7ac6d6915	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00079.nii.gz
case_00079	segmentation	209383	4bd69f065e4bafe049926c2e09eab607e19c353302b371411fc25bfdb6dc2700	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00079.nii.gz
case_00080	image	43158941	53d2195627a643feeebe65087baba6169af0f1707a26de5f7ee13943d3a4a0fd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00080.nii.gz
case_00080	segmentation	138357	9b4487b8f2e8cd821c417489586ce4a85e701698154294d98539ca45522c2bc6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00080.nii.gz
case_00081	image	60350821	78bc51ca3da42af7ceecc094b0709daf20925d8558a4175feffa31fdfcbcefc1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00081.nii.gz
case_00081	segmentation	223424	f05cb18fabc7e80f5dfdc265574eaad3f1870a66ec087a3ad28f0b7d55441c35	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00081.nii.gz
case_00082	image	57346577	321ff379b9ee8166faebe129184ca75f8715bed42ff10edd91001a8a3d3fd8b7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00082.nii.gz
case_00082	segmentation	179333	8276a8b0a4f610a81232617e59006334e6c901a6449d2d62127d2de9af6e0282	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00082.nii.gz
case_00083	image	38295161	7748e2dc080c6f9df479e7db2b03d17c7b32e62ee081741fd5d0d0786cc1fa23	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00083.nii.gz
case_00083	segmentation	163323	a53607023a9e5316dba7f4db377fb13849facc090130ede81980a1c10fc56097	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00083.nii.gz
case_00084	image	112681578	d2c54a087dc9b2e1229a326ed12055dfbee596cf2bc7ad8a247b9fde540738d7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00084.nii.gz
case_00084	segmentation	401040	c822dee77250eff4671bf97718a2d4f4989b39e2eda5e68878c7b8735138b441	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00084.nii.gz
case_00085	image	36294311	b875d694ee4e8c3e8125ee8be2f8d317af0dd4124929da8546297b3ab324772b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00085.nii.gz
case_00085	segmentation	128346	40bdd1b6a9f97cf856d005f0e0e59a9792ea99deecfd750ad51ad4c84446a55d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00085.nii.gz
case_00086	image	81962715	f05018951244e877cc6cb3aa842ed2287a32bab86b4cd5e8564e0ce6210930d1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00086.nii.gz
case_00086	segmentation	305725	9cb9b8b82a90b4d8a0af353f2e0e726f8fd1e64b08a798cc3e3c2ac0b2d75c8a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00086.nii.gz
case_00087	image	25303768	f4658463b4480b411d7b5e283294a828145acf95cef70f9e35f88161167a5298	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00087.nii.gz
case_00087	segmentation	98329	cdf8cbb56bf97708e784ae1162931ee0e563fc8d31861cd763b801f2a639014b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00087.nii.gz
case_00088	image	47969595	393684ed2220f0e030c09f3ed36e832740e5e9a8c98bffad7480e2d60e79d152	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00088.nii.gz
case_00088	segmentation	209600	fe1d9bd3e8cb581df6d06d33d76a475c5a0c12026ae1548f49a8e345ea17ada6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00088.nii.gz
case_00089	image	24690956	3fed1d4f01ff7e04d618fd48000937ea455aa32bc8b1d693697b2e258c95dd40	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00089.nii.gz
case_00089	segmentation	96362	03f278a2138392a16b0c036fb604d52fbc7ae0a2366cb0fed5949060c53f4cb3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00089.nii.gz
case_00090	image	35861921	5bfa1c15f922bdbeaffa7cd12c3f9f3cb7b6100db29ec7811ead40e977bb490a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00090.nii.gz
case_00090	segmentation	129973	9e564d8aed1a224fff4a815f3ef4f7eb8b7f099d7e5ebb95638c951a53433bf9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00090.nii.gz
case_00091	image	320735560	56f4a7e6531497523448ab4b2c23f61cdb907880161ca349e1cef88e3a3f31ac	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00091.nii.gz
case_00091	segmentation	989546	8bc7af7a8234c0705bb09b899096ce505b184c21f45adf72c610cfb2750f4424	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00091.nii.gz
case_00092	image	51076798	0e5beede1e2deec31a4bf8f6c0cd51c47e053c3a4644341a4178678be59131f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00092.nii.gz
case_00092	segmentation	201679	67bb898a1a97dbe4a612b9cf84ab91a418f08fda606fdbd047ad64891531a7c7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00092.nii.gz
case_00093	image	324309850	023bae3d476a6993ed654e9a7af46c9b659a6aa6903d2cb3829af5533e5ba42c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00093.nii.gz
case_00093	segmentation	1019837	bb37b246c3ace2280fb85d58699b86d764212ae6df17b0d2e79b6388755d96e6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00093.nii.gz
case_00094	image	17428859	48802f2c1d197f63c9be459d2402c9823ca9387ad4699e482ca3a2da4d8b2c6b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00094.nii.gz
case_00094	segmentation	103408	988638762c8aff27f770f6143f97749129e68c89750c3d847b72202f7a41a4d1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00094.nii.gz
case_00095	image	127649269	7908ff06fd6c82652ad1591b650c71529f616bf4f904fee75e6604dcf5e3a1e2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00095.nii.gz
case_00095	segmentation	425778	f51eefee3109edd55d816727bbe2a851e27bcbe5055112c1b97f1f74c0ae0fbc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00095.nii.gz
case_00096	image	303181978	d0570233199a3c59ff8e049f98732d3990564bf45ff67e28702709feddfd1085	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00096.nii.gz
case_00096	segmentation	912730	3ab8156c917b70b957710be8dc09e2d189299273371eb1fbb4806d91b2e2b1c7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00096.nii.gz
case_00097	image	35994201	78c2d1a5c4cf9d7f492b01db67dde00a1c106dd0dcde7d3a3fb66b790c58019f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00097.nii.gz
case_00097	segmentation	132817	0e1b59b3a54645b5c2ae1393b21b5c45ec49fdad7c02477aa7ac59a964eab850	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00097.nii.gz
case_00098	image	113065659	7b90ea447c544442524c07d999eebd142c95645f5e033e14f6dd138fdf59d3be	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00098.nii.gz
case_00098	segmentation	377210	a1be494437a341fde41bb160f5ceb328c85cb023990c51c05b8d2189594da1f1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00098.nii.gz
case_00099	image	44531186	48d45ad59be0e4ca0a51531a74ffdee5ce530119ce0acaa406d08d8a66062063	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00099.nii.gz
case_00099	segmentation	170886	d4e415781b2abd8fdfe1eb9a6bc1dc604870a43512e8864dd3c58792b3a2ab41	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00099.nii.gz
case_00100	image	217447355	2391adf361de6377cc4271e4180d2be74b052303833bd60ad26356be9cefcbbd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00100.nii.gz
case_00100	segmentation	617390	1bf0fd7e3ee9cffe3f8143aa0b34f9788b4c4c9bb5bcc154384c2aa4a22c3dda	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00100.nii.gz
case_00101	image	222646868	fb57e0feba61718803ed3756f5b880ea7be85e43d374e67eeafb22ec1bdf0ea2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00101.nii.gz
case_00101	segmentation	688420	8145b4285d6c0fa1708616e1b2c12a645b0232af33b033f984bcd3d5f14204aa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00101.nii.gz
case_00102	image	139727628	ce1f4182738e0cc5b36ad02efd719b805699b8f5e647e7ca12aac2fe9147c4da	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00102.nii.gz
case_00102	segmentation	526237	42116189e498eab82fedef18fbcacd6b9336ce8c79d7ab1966e533a68277d3f7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00102.nii.gz
case_00103	image	286971910	0b8a27bd9af3a52f55c5a17fee98043bbbcc997c1a66a356984d3d0d4ee66375	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00103.nii.gz
case_00103	segmentation	846921	e3ba11bda484deeb93c179a49eb96276f7faf7c7d425770319ecf65ef6475c19	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00103.nii.gz
case_00104	image	58853229	16b1c9224a730305486c5a741f8ca04cd82852e3ba14850cdc53fe07ec96ec5c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00104.nii.gz
case_00104	segmentation	172216	9252d28cb02ec1cd544568aa5452ca8c01a0ce478076bc84eab613e58d964ac2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00104.nii.gz
case_00105	image	60940672	e4d2fe3b5af221ccb405e8e892cd8c03077f926c156ed23b054c9b7b9badd886	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00105.nii.gz
case_00105	segmentation	217977	a73b144723ecc93fa0b7512bc66402af9cc174ae53bc5420a4538fe884be18f1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00105.nii.gz
case_00106	image	47144713	384c34749a9c849a2e0d4bf70335ba15696f79e53b136cba21f18d44d65e34ca	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00106.nii.gz
case_00106	segmentation	161015	a5972821fc98f57292a07baf5fa6c1098c567fe21e59b6187a46a52e516e15d2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00106.nii.gz
case_00107	image	34804025	1c1e8da81cb453a4a4da056f056302a97b51bd7f34cfd254d15d80d49540848d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00107.nii.gz
case_00107	segmentation	156692	d920c0a03299e78ced3392f7651ee5ab2a455e249979ee6ae95abbf003b0f4f1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00107.nii.gz
case_00108	image	23687235	543e722f44db15e7452bfcc39d97acf5803952344c7a3a7ae31d4c4577bd3aa0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00108.nii.gz
case_00108	segmentation	101299	f8a966ff97c59738477aab160c06e8656cccd46923db2163f96b820f2e448b41	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00108.nii.gz
case_00109	image	32402891	7539f075fbcc4dacec246cef7e0e4e879a06a93d2dd98bfebaaa6c8930895a77	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00109.nii.gz
case_00109	segmentation	165324	f78cc8c074b3eb508f631d82f32d742b25445dbcb55ea93470c9adc2dcf22ab1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00109.nii.gz
case_00110	image	16481496	e85aff843a39845a1a0bbe488ced890bb50ff3d098721d9bbf39d4d188acc627	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00110.nii.gz
case_00110	segmentation	70527	8c89ee548e1d0740d9643dec735859899b04a0187868a287df8d56157495cbf6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00110.nii.gz
case_00111	image	78705328	4acbc48b42d3a56e0eab6dd7b6537b819f58ffebbad6043b76245a75d4176cda	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00111.nii.gz
case_00111	segmentation	295519	e6516d26404d64e6b1b871e9da50a6b9f95dfefb4a654412d2fd9c1e6e6f05cb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00111.nii.gz
case_00112	image	55840497	935e66df54397a86338502072f2212fa25d51f648669e16ab7733abbd19d8241	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00112.nii.gz
case_00112	segmentation	227914	07f8ffa88068248c468a3700bcf52789230e3b60dab28c0078e3aaed6e662b20	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00112.nii.gz
case_00113	image	22107071	ea0aa6393a3bb4734c152730b605b595775646807edd953d36714b9377300e61	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00113.nii.gz
case_00113	segmentation	91901	44497577ed06146dff214ec9f1c2ae26db33707933a6a2a292b8bd0c413c76eb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00113.nii.gz
case_00114	image	147217241	09a8723f8afa6a7b811e32bccd8643b2c4fe9efd9f164085cde62757ab7bd0d8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00114.nii.gz
case_00114	segmentation	596066	5f0c49dcd0e47c63c3e03c9777fa7f2c2fcc3daf3c443ee9b721db82e939fd01	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00114.nii.gz
case_00115	image	147301746	bbf592e97e02fc0c611db54c53080d4ec35981f4185fab0e89994e4ea2a438e8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00115.nii.gz
case_00115	segmentation	465782	3d111017e695b8201bba7f9d29fd1ed9a189fee36eb6a4e103848c15d7f88ef1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00115.nii.gz
case_00116	image	196707070	525f60083da40205dda79a4cb1298b9b97a70f6bae26f20cd421cd67caf81546	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00116.nii.gz
case_00116	segmentation	604912	9348876d66f9728e2c7671e2499f823d6fc29b346a719caf95fafb00b9d73449	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00116.nii.gz
case_00117	image	30684538	37e270565cae64838698ed6dab7f0a23867901e69d8673dfa0c083efb36cb463	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00117.nii.gz
case_00117	segmentation	150474	af6976fc8fbef358b8310a931dcba0c64d33107233457748d8d8d10734d9c386	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00117.nii.gz
case_00118	image	303939757	46deca3f230a031ff2fb5d03b0922a382e67e5dde0c7d387c442cb89084d2283	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00118.nii.gz
case_00118	segmentation	931275	a5af02d1abf76156f3fd01338dd702ecaf5d2d780706c615d692a7ba2f303e93	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00118.nii.gz
case_00119	image	44352103	14539f65234f068b1e349e753b422f564198021879612264556eafc1e1fdcc67	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00119.nii.gz
case_00119	segmentation	161112	8faf602a877258cc6ca65e5d45cc9b27095d48712acd76b60192e5e6bb02c632	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00119.nii.gz
case_00120	image	135413522	3332d84141993f42dc03d7977a7bc81e199f94be54bb567ce09554ab2f7b5732	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00120.nii.gz
case_00120	segmentation	393524	82a79571775a8404576123f7f915f9e97eebfeb6108761d237f689de63d96f9a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00120.nii.gz
case_00121	image	35289520	3fe7568dbf5446747878ab9ce27760cad28b4cc043295457a5869469e34cf914	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00121.nii.gz
case_00121	segmentation	124241	62b8f9817b363234376598f40955423084e7fbf6deb0f73337575aa10237aa95	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00121.nii.gz
case_00122	image	25550117	ca3f77c9ac09245d7290364a08c9efea4d0e7870a93bb73ae8755c5bb1b34ff7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00122.nii.gz
case_00122	segmentation	106953	444a228d60a54b48ffb70387e6cd2db1b7bb5959a4361aeb01c11ad0705fde7e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00122.nii.gz
case_00123	image	150859371	be132ec55663137296e74c7e8f42df74a6676c347c05c2a0afb2b6c512dd31c2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00123.nii.gz
case_00123	segmentation	509299	61b189d18553a1a09c51d8a118e564a81f5117fb7bb6b60ecdcca1f9df2ec5c7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00123.nii.gz
case_00124	image	113055034	57eb0b37673a18c9d81257afb7c6e8a079574bf2ab8250eadeced0d915060a00	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00124.nii.gz
case_00124	segmentation	383036	23b7fb3ef3917a56eca8839187de570a7d27905c3410152950397a00a54bf5ba	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00124.nii.gz
case_00125	image	64037453	7025178cb9775c773dceeca725b5cf1354c9547786483bec867044d40ae7d069	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00125.nii.gz
case_00125	segmentation	240960	665c1a1efc7480707df08ff419e0ac9a1dceae0851935ecfd2d30ef555227744	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00125.nii.gz
case_00126	image	46555768	af6b9438df4a10b5561bdf9072b806189784e2ece7b38e2f1440ad2fdd51b9ac	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00126.nii.gz
case_00126	segmentation	170183	581389a2c912f2ddc7772cb322d51699ddbf629e88e00856919dade0c379b93d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00126.nii.gz
case_00127	image	41530807	bf94415e76b8bf0a28b5569afa08e402718ca902ad485d0ce4267b47919502f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00127.nii.gz
case_00127	segmentation	165871	2d06b18689d9c15942772f95bc9932e23d63a2d3f0a8cbc2e7dbb9d83348ada2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00127.nii.gz
case_00128	image	87078506	44aadb43ffdfca340cec735f990c875e407dbcee7a73d2bd184f401b7f6eb05e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00128.nii.gz
case_00128	segmentation	311297	9d80c8d809c1580d72ab0418873d8e71f312bcf6c0c85c01a4aa5d78c3b9364c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00128.nii.gz
case_00129	image	24054442	0d7b29ba62bab0a031a98fec6651230e7dfc8d3fc599e3d2201dfe6ce114f245	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00129.nii.gz
case_00129	segmentation	87291	9a3f1b86ff022171ce9d11aaac503e14968ef4ef31ffe3afdef1d9411b0bf87b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00129.nii.gz
case_00130	image	26677591	5d8c6a9f58ea6644879b2ad760560e743a5f86a4cb2b3682f5e1e0cbd4f5dc69	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00130.nii.gz
case_00130	segmentation	129963	285e5ada4a8f3aefb7a7698fd01ab953ae11c98baaddf6bb8b2a7bcbe9953325	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00130.nii.gz
case_00131	image	66542549	cb146cf250f9607c325ba34dd21d7f126121045dfcab4d409e6c934de997d82e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00131.nii.gz
case_00131	segmentation	251633	d1288e0b64d7816851c20b36c5e4955c489d5ba6131c7c6b78466c26e070645e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00131.nii.gz
case_00132	image	240243768	8ed0c03970a1e6681a63f3de19d09e45ca84d90290877baa7ed6c82a9efcd4fe	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00132.nii.gz
case_00132	segmentation	733135	39ea963c5373f6d3ef42e727c9299e5a689709fefe7bb6ad475bb212578adcc5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00132.nii.gz
case_00133	image	73000317	5d91b6b7ececbb86722b23911236cd6f53d983b5dafa8be556e3254c692c88f0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00133.nii.gz
case_00133	segmentation	257238	1b2e916cf0686895bf9936d50580af4d66a4cb133ef274a04dce98aeec507949	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00133.nii.gz
case_00134	image	24422346	8b4cf8b847bce58032c426ad57f0fa2a942eba2883c2082ea0396b88fc2df9f0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00134.nii.gz
case_00134	segmentation	105199	5499a1868d2e0209db060a15844493182fe8af1a73a0f2483f8bcef17933d274	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00134.nii.gz
case_00135	image	269658966	92e38761c584811e0904318abaaffc45fb7812aeb5a84df8a24c35796a1a401f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00135.nii.gz
case_00135	segmentation	904085	ae26e6f5e70e3ab4ed196efa0443f03c2e301124e1d5302b69468ddbacbcc46d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00135.nii.gz
case_00136	image	51986338	3e294c34dd7099d849f8a7d5259864d4f8a40a467f3dcedfc458ad93a6c4a6c0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00136.nii.gz
case_00136	segmentation	163754	056369058d521825de0f5c198cddaaa8fde95307221d452b03658ad52e7b22f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00136.nii.gz
case_00137	image	47700994	d6a732fdec037453469c5ba6f6ec0bcee49130530e9c10dbfef2ac21d36ddf03	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00137.nii.gz
case_00137	segmentation	180471	e98f438d6c82e2a1b86006767b053f971aae4e5a09582ec8fc2be5b45a97e7fc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00137.nii.gz
case_00138	image	40601700	7b7c931a5f23f2db73739064109ebba8c206c7e13681138e8adb74ec7c243b6a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00138.nii.gz
case_00138	segmentation	158968	95bf0e184acf6ba60dde3f8936b6051c4414694a4ae3c26d16ace7c8472e2bee	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00138.nii.gz
case_00139	image	48587729	5041d20b2db1f2e09b1711707be1df1e95ba3c9ff65d5fed6de537dc1e023e21	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00139.nii.gz
case_00139	segmentation	210943	ea76984c7c8541017eedfe6984f92885be101d0067846cf86ca8fe240157ed37	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00139.nii.gz
case_00140	image	129580387	34d5740ad5dfa4f638a531e12b880b99add5270ecbd1a7ac42c98a465b8a4142	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00140.nii.gz
case_00140	segmentation	406649	8f0eb743356411643aa5883bb7bf6beba028b133a6a9b66c40ad189e20220d89	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00140.nii.gz
case_00141	image	271847392	02595d667b7ac5c714200dca7ef4bb6f83d95e9c3882d5e1f7605191986097c3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00141.nii.gz
case_00141	segmentation	835002	3d7411accad7742a4d1ed0d7c5fbf69c9dd76d723d91932be1273abcefda1d07	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00141.nii.gz
case_00142	image	119786560	d11ef32d69bbb5d455f825a9ae7d36d9d5aee00e4aa0ace4b0aa2c6d0b96c419	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00142.nii.gz
case_00142	segmentation	403910	28f95cb2853dafa4ba5d27fb6811c11179794a21a281f1673526158838764f14	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00142.nii.gz
case_00143	image	39109278	b9780eb096da190e3327b60d85eee9d6f11ecba54613d4c62254fa03ecdcbc6e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00143.nii.gz
case_00143	segmentation	134506	1334cb62dba0e47898a894dee13a3b3c91a0e05bf630990428110cacec25e4db	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00143.nii.gz
case_00144	image	86697738	2d4b5d233f28e4a9bcc01e9928fb51926604177a41985bfcec8f9d3d2d2fd779	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00144.nii.gz
case_00144	segmentation	297069	2869f6a1711f137661cd288898882d13253719c00baba09bd376dfa5b436452b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00144.nii.gz
case_00145	image	32713163	9657b6eb9974a9be4f45bac54e877c360ca5016cfe3ae09a4fa4ff8e645d319b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00145.nii.gz
case_00145	segmentation	148572	5ab5418068d314121e925ed8a70d60f4e953a27a1bda24b163db2e7aac2b2c76	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00145.nii.gz
case_00146	image	281605168	1d94dc738a69e50c72cfab4203452eaf4771b30ac17ed818e9c0e3d140b155a2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00146.nii.gz
case_00146	segmentation	865323	451bcd6c915221c350edd5a96dd82a1e1678ead19467c15f1fbe2c8be35258c5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00146.nii.gz
case_00147	image	28532981	0f37a518fa0141a332f6230aebe7deacb55adfcb32d6a40c3b6171b0a007bc05	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00147.nii.gz
case_00147	segmentation	112359	1a3842493a08a135056de285b1c59e9936b0a66f4f06bddc0059fb53c635f659	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00147.nii.gz
case_00148	image	15411691	b4c93d60edc9e0ead4bab9390370aa379e3bac6c4f25e9a746c50de30596efac	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00148.nii.gz
case_00148	segmentation	69162	cb5326355c19aebe72b348dd9c972acc84f9aac512403488eabb292d9ea35265	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00148.nii.gz
case_00149	image	47822495	c4fd6735554a08d6c10242c98116d054d16b8e8483cad1985ee661af2b4ad204	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00149.nii.gz
case_00149	segmentation	189816	5dc980b777d260842116925f7bce5f3d3ad7de65a3a259dd7df0f38f86214aab	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00149.nii.gz
case_00150	image	43249791	35b19b44daad521f9de69b0eb9e6f5fdaaad33af9139415249247dc23d4e3d16	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00150.nii.gz
case_00150	segmentation	198926	5ef0d1665df7ed7d360c63e1c97e8b0ab3add1591f3af55d8717bf0bde37f9b2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00150.nii.gz
case_00151	image	495993341	26f294ac7284e56a477e4a05579729cfd2364d39731c7cdf5a9e85a53960b7af	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00151.nii.gz
case_00151	segmentation	1420778	d7d9096fa6e75f1a4e22bdb91f6085ad544dc58bd3af0885155463dae0213bf4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00151.nii.gz
case_00152	image	19368559	b407f2d86257173d4ade726b97dc53853a80d88b4226cfde330b56d1fd6da71b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00152.nii.gz
case_00152	segmentation	75822	80a75952a6fb8752179dee5483ac0bec3e56b62a226e2e5c420e6249344056ea	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00152.nii.gz
case_00153	image	35727661	b26b7c7e247f7a36939919c6608c7a6a3e240bf62cb9d614a7f57a213b9e2abb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00153.nii.gz
case_00153	segmentation	158180	b29fc296a036c0af702fda9c02ad67ca52dfbceaa98f12f6474fdab3950a972d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00153.nii.gz
case_00154	image	185019263	cc58b880f22132050d2ad632e2032460b63ca37acca746384e5ccc2b9de39b38	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00154.nii.gz
case_00154	segmentation	649553	a33d2dc2d39ed7850c91300d5110bc34d642e6e80a2f356ced8750598503b5b2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00154.nii.gz
case_00155	image	235923313	8f5e3dd8e441e1f81a16ff3ff6144973e5e9056b1ec3c79ed31de2d71120a296	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00155.nii.gz
case_00155	segmentation	747460	6c10e8f1d73adfa32b17a1eee92bccb3d3764d787e7e84d827453dec565c94db	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00155.nii.gz
case_00156	image	363352307	a5424e47cd0a55d041abd22636ebcf01c718e9df918db967f78fb91a98a7c90a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00156.nii.gz
case_00156	segmentation	1231774	509475fb6f411c3d5e88bbf316799b4aeb5570e12c71d85b9b0feee7377b8331	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00156.nii.gz
case_00157	image	231036073	777b7014743ac0e78b88f33e7c0af01a89403abf7980c1f4aaae8417e011e995	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00157.nii.gz
case_00157	segmentation	739219	147e3ad470055ce7fc373a89e2bdc08b704011be1faa587b1b75161afe6eea8d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00157.nii.gz
case_00158	image	345812852	e2abf83f652dd3667b108c84521d30bcf0c55befaa9ecd734128902eb51bde59	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00158.nii.gz
case_00158	segmentation	943458	5adc52d4bb046ec20873beab467669b4f8337d592deae156ac6545eb3a301467	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00158.nii.gz
case_00159	image	321877085	8ee241aaa90c58b817bae1511437b73dcde02f5639e57acb0867eebb75dba233	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00159.nii.gz
case_00159	segmentation	960554	7c4338bdfeefacfd67d1afde40a6d7b922ac147b136bf736cc7e4897b9c3a35c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00159.nii.gz
case_00160	image	196732920	94b8a15327097349f4f5afce36c7d0f5228dd7e7045b07c6325930a79d5ecd9a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00160.nii.gz
case_00160	segmentation	715981	571e0b575313da33e5c567fd1bbe381d03be5c9b80abbf951e4eaa8f95f7883c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00160.nii.gz
case_00161	image	25615080	763b5306cfd062bc22a169f37c0917cf639b8534e59b748def6094e4e7ef6281	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00161.nii.gz
case_00161	segmentation	99579	d22094e5d6cb55179595f950d6b1cec0cddf5b4cf10bb1eebe2143af4c579d65	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00161.nii.gz
case_00162	image	45384396	1ff33933b02751241021af304a6e9c0f2e937aa9ed09f26bc62cb0c96a316444	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00162.nii.gz
case_00162	segmentation	146613	50b7c43137590d14d3860a777a17bdef13e065494858e66782e594423a065a60	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00162.nii.gz
case_00163	image	40205054	9adaeba960c7ecdcae5a2fb0ce0249c71ee54ff2d9760f8470f693a0d0489957	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00163.nii.gz
case_00163	segmentation	160825	7ae49c044aa9ed0d2c0037e824ff70f12ae3e5659849ef465a768da1760a7e86	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00163.nii.gz
case_00164	image	44482192	fdca69677341cf4750386d3c2effb407594504760ea26e5e1230f02988c40bf9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00164.nii.gz
case_00164	segmentation	142250	4f494983c28cb30508b71469c8a1bccbf01dc5148726389cc1bf46a0efcb0ee1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00164.nii.gz
case_00165	image	283401628	d4f322c24dde02a115754678bc3b8761512fc83ced045b4b26c0efdafdf49889	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00165.nii.gz
case_00165	segmentation	891224	6c44cb4b5faf63a19d9000a0a1b16ac599281ca5840a203c92c023e5d6045885	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00165.nii.gz
case_00166	image	41559822	83ed3f82c0aa87821882ad9290d5507389caaac175c6443f8d4f213689458fbb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00166.nii.gz
case_00166	segmentation	184536	d5d53bd00bd45c233801c07fec06d554b48a597b051c829ead03ddcc84dfb05e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00166.nii.gz
case_00167	image	41129566	41bf260c248a2e0f49f755e6f25c8850b6d8b8ddaef7ea34df1157dc33399ba6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00167.nii.gz
case_00167	segmentation	152384	d914b5628bb4c8795aad3f29af629a3c80ebd427f995ebc2ffc1dd05ffc76fd9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00167.nii.gz
case_00168	image	32048051	0eedcb23583f504986afdbfbafa88d6a025591af407b3a331e81ff5b2cd14fec	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00168.nii.gz
case_00168	segmentation	119995	66a48991fdca4b75e46a6375f5c0af4367873446d064404950be5293e25ef43b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00168.nii.gz
case_00169	image	42056633	9bad338921a2361779642c080c0faf06dff4a0c4003ab77eab1c1c2364145d94	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00169.nii.gz
case_00169	segmentation	156142	eb08c7e78d005b0d2ffdc12974af73e56fb6c80d3e5618fa3c5116e8c60f012f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00169.nii.gz
case_00170	image	130206791	d2bb42b0dcb29cf3b343944b0c47a0af83b7cee5a2e8882b901f5e52ef51679b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00170.nii.gz
case_00170	segmentation	449026	7edc29db335b13e3ff6c693fec8469af2d3495ea265c085640b98b24896ead1e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00170.nii.gz
case_00171	image	63813232	de426f4f308eb551d44caeb50bcbbd671b18324dc6b2c81c5200dcff4b4e3c11	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00171.nii.gz
case_00171	segmentation	194200	0dd2ca0e3d650fc1d0518752a985f4d4f14093d46816bf9550d1ae5d2e297ee4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00171.nii.gz
case_00172	image	37626783	1440a94e1d52368c97b87c04ff8dbc8787ad57fa313fc12bce625e6834703b34	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00172.nii.gz
case_00172	segmentation	169233	9d4a9fec92dba6e3a0fb510cdbe765ade7a5945c2375ebde88875ed2072f1f17	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00172.nii.gz
case_00173	image	45244475	9134f2224becf6351934aad08aa1d9102882da585c4807606ad235e5251d53a0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00173.nii.gz
case_00173	segmentation	144293	0cd3908c94e520f4a5de210e267148195743ae75875f3d33a864f8320e07ee06	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00173.nii.gz
case_00174	image	31894170	04a27e7e46effcd43c912d8cef93eab4c91b1cfc76a3f3c5be2122c50f4a0b89	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00174.nii.gz
case_00174	segmentation	124895	61f3e94e2d9cf824220f85dc444660a14e950cb0cc18e5584310dad63a07ec05	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00174.nii.gz
case_00175	image	54239063	8542e7533ed4a498b737e04926e7af31d11425bd7ee3a15932ab9ecdbc06ebfd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00175.nii.gz
case_00175	segmentation	173411	14bb15c94e34c830a1bcde5a02c6156cd34e262c7462c613908264c5a684c023	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00175.nii.gz
case_00176	image	41461140	8bb524fccfb6cd242cc4330a0a71d854f9ed3c26f8acd857c3ae148f559b160d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00176.nii.gz
case_00176	segmentation	158340	2291bda3a9fcf4bc834180422339ca56137cec18e3b94d6e411a4ac0af640f86	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00176.nii.gz
case_00177	image	42626421	50e24db7ce59be43326851c1e9b5a9627a6c0f311742c3795b9fe41378b35fb2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00177.nii.gz
case_00177	segmentation	133293	e66f334cbef5b2a9cdd5aafe4830f1a389ddbf551266252d6b3a4e500d4a8631	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00177.nii.gz
case_00178	image	35857922	bad36ed206f18e17bd51e44d5a2cf6cb68e236e6e6f72b98f2c26ebcce449bd0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00178.nii.gz
case_00178	segmentation	134233	5e3ce0e6ccf522f198f2ac7d04e744bc20bafda12923ddb3a188988a0e77ed50	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00178.nii.gz
case_00179	image	49463821	990eaba41833f11e9d627a47f5a02efa77ada784f9f565217f1be3a82a2c7d9c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00179.nii.gz
case_00179	segmentation	170102	bbc2b13aefb9427ab05408d63edc8c73fe8cb8e6ee6416b7d92382daad9a5169	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00179.nii.gz
case_00180	image	66993063	6f1734503a768aeeb1911bacf82e62fc5c4822081c8a964363f163a57d5369d7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00180.nii.gz
case_00180	segmentation	216604	9a9df51dbb56b75d67b8fdf6e7214965065d0e92d1ec6405a9215050f1877dda	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00180.nii.gz
case_00181	image	49292022	c3af31c5f53db0dc483af00e3c41ad76b43a23d1c8c7122790e9be594520c973	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00181.nii.gz
case_00181	segmentation	218334	7f3fbe978d45113c1031939d9f09205435ecdbbdd9fb375028cae5dff8caa1fc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00181.nii.gz
case_00182	image	38436116	212ac0948ec7cc787e5748a74e2390a3e14cef1351f56c163c17880116ae6c61	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00182.nii.gz
case_00182	segmentation	140821	6b45ba6c4a7e6ff6dd2c23a63ec2960bb7d2cacefb997f4d433a599fb3c93a4f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00182.nii.gz
case_00183	image	99950548	d5bc20c81c4eedfc6898e02d3078471b37fceb215b1121211d8541676bfe8498	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00183.nii.gz
case_00183	segmentation	319067	88fb7bfb964bb81c3dd5a896cfe356465fcb842263433a8f1f9b88b1dc8c0d50	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00183.nii.gz
case_00184	image	70271243	d65b497d685e52ba8b87dcfc10a0fa4b31ad17023510f3e0f7a2bfb61f0e3873	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00184.nii.gz
case_00184	segmentation	235413	609bedbe1a91014c12d9562257a746c2dc50ccafc3754e7b572b4ad7d27ff6b8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00184.nii.gz
case_00185	image	99730672	8d69db0601b8030a38f399c3458e05543395f2ef68cd4029696f02821ec503f1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00185.nii.gz
case_00185	segmentation	359254	e692e9a414247cc93d247caa15938e48d5c6451c548dbcff9d594725c53928b0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00185.nii.gz
case_00186	image	68574602	8ea25ef6082aca0b95ed4b06b968ec155a736bbe5711d93bfe772d38a0676768	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00186.nii.gz
case_00186	segmentation	229815	59d95f63feed423fd244530eedfdb82805423a945f7eb558779fe79f4934b9bd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00186.nii.gz
case_00187	image	41465147	56e4971bfbc4afa83efefdab2a4e316bd3af5e6cd0f5a9049982c83cc4833f1e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00187.nii.gz
case_00187	segmentation	156407	072802859aaee0af84f5bd657e9e0b0ce445cee2bc74ee8fd3c9bb069921b859	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00187.nii.gz
case_00188	image	173648780	bc7663ac0340226690949107d1cb2b735cbbf7d9a2275d3456f408cbbccb81ef	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00188.nii.gz
case_00188	segmentation	533068	c1e532388aed3c4747e56d9ed23723e2c7f1babf154594a3f7a365e29c3a537a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00188.nii.gz
case_00189	image	69794230	27e70b8e75d4151a880cfdb1d18e9edd79a1e328501af744b8733e1cee6a37c6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00189.nii.gz
case_00189	segmentation	241321	afbe79d87cf91ef64a85e4416a8ebe3ff1b9f2025eb1942a7b3cf2f15a8b99a8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00189.nii.gz
case_00190	image	80969548	5ebf726ac1dd126fa5b382041a2cca35fd1a65522394ae24770a1cd30fe14caf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00190.nii.gz
case_00190	segmentation	273642	d084da4f5aad738867884c4d3d3b17ac007e5608894cd166dd1ba085c6674e7a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00190.nii.gz
case_00191	image	146751277	0978a60e2575988d4fc900b094d46e0d7a56ab9e199961aa5b407ed67aaa08c6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00191.nii.gz
case_00191	segmentation	498862	4f44859d48647f18c2170151685b5a17f5b578e7ec93ed3093c4730ded2fb658	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00191.nii.gz
case_00192	image	56112884	e8ecd933dd2e7cbf924af5441aeb6e6e8876c35b4dce28957a6b53e85cad7d3b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00192.nii.gz
case_00192	segmentation	241637	53db288a6ffc5e07eb62ad2164c0cc52134931da18b038f172127e6cc0d865bd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00192.nii.gz
case_00193	image	94515487	89bb0eb8074c3b343e2c7b5319b9f89db590a0dc1544c02c1e2ca81074c3dfbc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00193.nii.gz
case_00193	segmentation	350280	b1c590613edc686247555872b42cf28a04186c046902347c8fcc640433d90266	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00193.nii.gz
case_00194	image	56502711	7ff7425f0749f7aafc5587cc4518539dbf58ea8c15bf5a2328e2b00b0d9c550a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00194.nii.gz
case_00194	segmentation	189351	f5f5896e98b512f0fc3ee83e3ffaf571b69817d60f6fb6aa62e4fee353688b98	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00194.nii.gz
case_00195	image	47638710	c86e5275d7fb29dfe82371b8a0283d71572fdda2db8c032ac6642ebc6c81fe69	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00195.nii.gz
case_00195	segmentation	166827	f571ff6a0ab4047aaae660206e435263c6f1147c76ec1545478f711ec376c920	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00195.nii.gz
case_00196	image	80043050	04ac9fcdd99588e040a25542709a87cf959268e1c8fda0266344bc2160b8fd12	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00196.nii.gz
case_00196	segmentation	263775	7616ef9aa683d2b928faf01e291552691d33e70b8d74a0834d91d3f144c97608	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00196.nii.gz
case_00197	image	71480975	e98863429d6a4b036dcf4c3483fb5b4a2e009ddc6390d0da647424772b77b9b8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00197.nii.gz
case_00197	segmentation	252305	44556aa49c1005e31f44d513ac0c025079d109dab5e1d9b9af90f39ffff67156	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00197.nii.gz
case_00198	image	130795162	ea3140416a4eef35fa25a3051eacec1acde2b97180d3d247113cbcb6db81edf9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00198.nii.gz
case_00198	segmentation	382511	cf99bfd320c758a110437ecfe5e3a802ee89cf2ae0a0d344a07b5aa7cebba663	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00198.nii.gz
case_00199	image	48897494	8005f1e4040b41d78e9c926769217bd94b75c7c40db8d65db96e87c9605f2593	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00199.nii.gz
case_00199	segmentation	220427	c06fc7c6f7d1e2fc956df01be05c87b827e0307ef716ba8edf56881c87f3f169	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00199.nii.gz
case_00200	image	51449540	a442caf95f7ccd561c502176c6ec759b116a32a604c49fa06633d506d2bfc5ea	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00200.nii.gz
case_00200	segmentation	185541	4816813092810022d2217cedb92f9ac35248009d3f0ec126785f860d79e69e0b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00200.nii.gz
case_00201	image	47090706	62e9e470bdead6a6bbbacbebfb7669055e2ebb3e8a1177882fd028e0c7ec1623	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00201.nii.gz
case_00201	segmentation	191060	015b44260e5fd80efc81d56267f3331fa6c9b75160ffe67cdee9061d41938441	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00201.nii.gz
case_00202	image	145466695	f56e642d587bb2ac75172d9aae45c2b43e1f496424802e7ca261e672596315d1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00202.nii.gz
case_00202	segmentation	524353	2fe72013764b9c27ad2858f948d226b303a49ad7c2fd830068f696047b519466	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00202.nii.gz
case_00203	image	295206306	db1ace382d198df990ea7bc6131aa03cf4a446eb5692e1edeea9a59f77a27c80	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00203.nii.gz
case_00203	segmentation	802889	4c1018d71b2519c0d8a9d71f580a7d8126eb9a5e8f768413605d5c1858be2520	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00203.nii.gz
case_00204	image	31576322	2623b07d547eacea478765df84cb6710e0460248a32750a81cf6d212b762e93d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00204.nii.gz
case_00204	segmentation	139953	ea39fff721040121a20f40e6d8df791677e7c0f3d7f2097b095fa11f49e66a59	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00204.nii.gz
case_00205	image	44138736	169aa122b7169223c28794f88d5c3f5cb45b644676ef1b614d9101f8381cbe2c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00205.nii.gz
case_00205	segmentation	136205	e27486518488b19633d1ae6bcd91275f59e2217199e48b5604df9ff6cb54e325	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00205.nii.gz
case_00206	image	22988003	573e65adbcd6a8b709e83bff588fbad8cc4641a6fbab422bad1d927e4772ed53	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00206.nii.gz
case_00206	segmentation	96746	661474e5c37df6dbc2a2063e5cdcf1e2d2b7519e50a2b40d5d1521a13dd75c06	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00206.nii.gz
case_00207	image	68719670	c665dcf69398aee4644526535425c29fc1f0ed2ba0cfb4e9e1d0c6ce0b8f3a9d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00207.nii.gz
case_00207	segmentation	208354	aef01ca8ece739bbddfcdc7b154f021228ad2b1e689b75720668533eeb580d11	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00207.nii.gz
case_00208	image	39898045	99cccfcb17a7379ab67ba264c44d83015fab1254b27a090f991177a3e629d270	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00208.nii.gz
case_00208	segmentation	130024	9295fba3177b314426b474e3a79c5d2f4a6eb3b069b6c72fc161387c78a8aa35	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00208.nii.gz
case_00209	image	50270697	b050ce75765edefab6097dcfc87568800f49d03e09c306e3f4375c66eae45e41	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00209.nii.gz
case_00209	segmentation	158556	d96c7e70ba4880c259a834c6633ae95bc770a8f66dcc553007b6b0713d884ced	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00209.nii.gz
case_00210	image	18415929	9d5e9482645ab337ab57c91d16d52ddb1ee252713ea560f2838b06f31002804c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00210.nii.gz
case_00210	segmentation	89336	be67098a8e82623f4546f08f4abe94fd80a3d4c79acfd4d0eaacc8f143d73a53	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00210.nii.gz
case_00211	image	42867523	bb544af082768ccc80386b71e49267801642f98b848d7d9468fdf84a3d69b06a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00211.nii.gz
case_00211	segmentation	210560	0a6f2f0dfcd56d18424bbcd239fbe1479d6cabeeb802d0dddc0711a9d97dda6e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00211.nii.gz
case_00212	image	42084365	1345506f81d2a95291e7d6f4d7a0b0d645e714fe5073593f4e3202ef72a38e7b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00212.nii.gz
case_00212	segmentation	150340	105d2b6b80d562f7f4c396445519e229096d8359faf8dd8ee19ecc59654320ec	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00212.nii.gz
case_00213	image	290128723	ce2f5d3bd608c509a043cb0f7323e28754851ff6ecd3a9bd55fb4219733e93fa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00213.nii.gz
case_00213	segmentation	798378	5694d5efedc507939e325de58b3e84c230161aca37776830a659293baa5a6656	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00213.nii.gz
case_00214	image	197325363	394c25d624096ccd40099b9a2dec982532f5dc5bdc854ce4f7c28f7c8e653cf9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00214.nii.gz
case_00214	segmentation	658321	49571adce018a4f8d9e304098a33c4a3c867d64833ea1629ea4715d029fae89d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00214.nii.gz
case_00215	image	32360915	f783bd4677a2e3b9e6c39ed9b7fd8874b2bbf41a1b63bdd8d9c5e06621a2fcd9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00215.nii.gz
case_00215	segmentation	124038	065906f68d94780e8d172eedd056ad07f0abcce22ca457c4b152bcf427139911	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00215.nii.gz
case_00216	image	318160610	d3daf83ee260e01f3707f999c7badcbc2a3edf77975f568de0e714a02eb50c5a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00216.nii.gz
case_00216	segmentation	1028637	5961c7871dc382f7855ad59bff0499b00a0e64aab0829a874aaa623638d0feb9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00216.nii.gz
case_00217	image	66593365	ca3ae3118fa52613f02bbd2725aef177b4f2e868d6c7bb53152af09181007644	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00217.nii.gz
case_00217	segmentation	220255	bbc1f3ff26df6c3d7670588c507acc9e72d5d468efcae1b0737212c7bfd4aaea	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00217.nii.gz
case_00218	image	73150864	1d57ca21412fff2729ba969cf1df176ba51064541090b3dd4615914f5b5b3f59	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00218.nii.gz
case_00218	segmentation	245366	a34dc5040bccb92c3fe52e009a962a72265f54135e7bbb2f5b7bb3983a1b5927	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00218.nii.gz
case_00219	image	74788567	06084ff763727b80a64956e7802f5b92728c9107460fc13cfa454f9e8606c3a0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00219.nii.gz
case_00219	segmentation	262565	17df52ef9cc281c79ba400cd5f1af613f10253f05294f40f6798d9963e3b2b6f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00219.nii.gz
case_00220	image	86327855	fb788c6099617cf4d95e8c3a791fee7177f87281ce7a7bb1399df1c08f22e041	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00220.nii.gz
case_00220	segmentation	271664	6f402e7a40f543ef71188f23adf57e124bd627fd741bc245dad33828c6fba93a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00220.nii.gz
case_00221	image	68228113	f72c750c30941d489a49d8f74c96d6a375928095e2bb494903a8c786a8bcfa41	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00221.nii.gz
case_00221	segmentation	237098	cb774fb4f147458092804b8fd5e7545d6e441ffb2a45a11c03eddae1deda42b4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00221.nii.gz
case_00222	image	44420089	86f1ef3cacbb9a95fdea5924d7eb58a645af359ab87bdd5970ede5de4a9dd78c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00222.nii.gz
case_00222	segmentation	156736	7258753fbd5f319bdd0b7b14d0036e8e519b775f87d8adc10054c2504de781d2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00222.nii.gz
case_00223	image	175070282	dbfd7951763a97476b4d708d8ec993e42130f35bcdb60233a5814fc50df8b497	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00223.nii.gz
case_00223	segmentation	667471	af917a0b6937df77dc7c2f582d0d08e5d70aaa2bf9cad00e2cb11fac529728ae	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00223.nii.gz
case_00224	image	41716731	063ed96973a0f2be363b6dea4615723131cd67fc15af692e7aa7780252599bee	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00224.nii.gz
case_00224	segmentation	163950	5e18defd971848f0f2d08c654206095209e033a26a021f1f96d5b58441b05dee	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00224.nii.gz
case_00225	image	51437705	b2b12c64ba14e33f76ff714b5bbc354922a44c89e23f35738d00b0a8ec86de81	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00225.nii.gz
case_00225	segmentation	171243	1c0a0511b6ab37797d9254792b8c33e1afe799cd3c447e684064529273554aed	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00225.nii.gz
case_00226	image	62450432	acd45477086ee042edc03c871c3b4f42954bcfe8421c9209f803d9dc4e348d3c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00226.nii.gz
case_00226	segmentation	228032	a45d55af354d19ad230946415a0c01e55be74643b7c8d7bb254bbf7dff4083e3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00226.nii.gz
case_00227	image	66362147	e414d7fbbd61b78fa2f314fdd629f33acacccac2a47d7b19409a8634ae23cb9c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00227.nii.gz
case_00227	segmentation	217856	4c6822055363acb3f7f978039d4d49514be50b2b98cf53355b3201c0e38737cc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00227.nii.gz
case_00228	image	42517543	78d5ee0c1172d0b23f955bc59fe98e3e903371a7c117d6067a5a9dcd8e935208	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00228.nii.gz
case_00228	segmentation	196232	e661bfd92e358b0ff7dfc5bc23427114be00d7854128ef98b0f3613c8f969a62	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00228.nii.gz
case_00229	image	124090788	0248e817b15d63d8b3dbd72417cee8edad6919913582f0e654489c755b76ba5c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00229.nii.gz
case_00229	segmentation	380964	0abec1e37ff22d6e06fbe4eeb66bab093b7b502481c0f448f7ba5945a93b2f57	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00229.nii.gz
case_00230	image	46409362	d687c3f8fbaa3a25be07132afb2b412b14b92d5ed54431acb976a118e8fdeb9b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00230.nii.gz
case_00230	segmentation	154671	5db9d0d23b9c2bb51aa6a4d285336cc387754a5d89ddb90f13656087d97e650b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00230.nii.gz
case_00231	image	72366409	c009d0ec0b3d55a6aa1c369d1126a97e003a9ee7dffaf03d31db6d7cf929d998	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00231.nii.gz
case_00231	segmentation	251203	3ea73114a6d0b75c914c0cb2d36fce5a5e153b61571e1c44cafe6014f4986809	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00231.nii.gz
case_00232	image	44837071	b2e21106757a64973685ce61c52430e9bab5fc68df110fbf5c01751a051a6031	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00232.nii.gz
case_00232	segmentation	156894	3ec42ea98d43ce1a562b7e03958dae3bca7090e5f0a96adbabc4ff6efb41ea4c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00232.nii.gz
case_00233	image	41518856	d2ab2d5c38176ce4a2217ea8819d37f9b13356de4f1a2324cafe550568b4cbea	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00233.nii.gz
case_00233	segmentation	224823	35222af15ca3e855c5b5a7646ed327d94c84e2f088d7dd2b781dc26614d5a4c9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00233.nii.gz
case_00234	image	69470513	d728c1be0d14f82f30a88899134849defd84b3597593929fb470d842a73e66b6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00234.nii.gz
case_00234	segmentation	222462	6ce49ccc7a6d7be081a4a23cf11a37a31c25c4fd1dc3b3700db7cb3e3fa206ff	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00234.nii.gz
case_00235	image	79819592	7591b093b9661d49a7a39399b842f132007f0ebcbe605c72e0c1914f3b6ada43	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00235.nii.gz
case_00235	segmentation	272057	3b18cb823564a0da84eeca7a87531e46cc8096ca1a03efe27a185c751d186684	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00235.nii.gz
case_00236	image	28360138	b5ae3209cd777f6f4562a22fabff2d009010b4cd711ff49316c842513dc1e802	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00236.nii.gz
case_00236	segmentation	118224	07685d7cce3239ee30e6fdcfefa038ac92228ce5831656020c660f0f50aae7d9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00236.nii.gz
case_00237	image	67799083	e0030fd260c252e6c143acde6d8a65656f8995fcbc773d1fa91750ed558c4a37	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00237.nii.gz
case_00237	segmentation	259381	a8bc9354c9c25e24878d4a04933bc5cc02d3fa951d298410acceed4ee94a7001	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00237.nii.gz
case_00238	image	46579368	9b2bf6749d2ce203a7e196f130b865a5e60159ebf70a9a9ad4fca6bbdc2f86b0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00238.nii.gz
case_00238	segmentation	155434	7796c7adb44dc6f39d1d2dc0209e6f4250c097caf2ac9ce3c7798de760550e7c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00238.nii.gz
case_00239	image	42971344	f915dba34c61f9da0128a1c160c35027ecbfe42c9165e7c983d1055658c51bd2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00239.nii.gz
case_00239	segmentation	148994	723c4b0598ecd2122bcea87f4893aa565eab9127fefc5830f660a9e538c682ed	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00239.nii.gz
case_00240	image	80533509	157b7ee995e75ff35fef5e779e2bacfcfa7bd0ec3efd3ac9be46c49ccebe832a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00240.nii.gz
case_00240	segmentation	282770	cfc08eb8eec78966e30b45c2e702ec1c72f4c461061c6638b09a6d380fdc4c14	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00240.nii.gz
case_00241	image	47966927	a8a9605e6300b35c7f715234f4061f0a5931a299149cd3e13ee077bd60e4b5df	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00241.nii.gz
case_00241	segmentation	168863	fe90527a8b3f7cbf1aa5d79dd8d20a98714269da8750a614dc2f8009b50b76c4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00241.nii.gz
case_00242	image	39443969	ea7df74c9b3f93ddd8c71ec8b7a090dd77091e719a310828d2bf28618b515f56	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00242.nii.gz
case_00242	segmentation	128545	c3a320d6505fae6286ce21c5651a4e9dadc9a1eecaf2586c177bf95a9b5b8f4e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00242.nii.gz
case_00243	image	41461643	16cacaa27576db47e7f2278b938141a518204dc509b6bb6bbf3a4db29fdc6aca	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00243.nii.gz
case_00243	segmentation	141946	8cf0f03e23dac0eeef496706727dfa608d80d6003f6bab8ac100d235dec69daa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00243.nii.gz
case_00244	image	35423582	18b6e92995cbd4906c00b7f85b6c542fb776c72d0ceb7e39b9c813a3712a7337	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00244.nii.gz
case_00244	segmentation	172298	b84cd0dcdb8d1723ac6aa1fb90db3c944c82e18a5f3778163b4c2bd2c5266dd5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00244.nii.gz
case_00245	image	59856117	576bb39ed6c57983c95abd41d22f6ff95dad796b7598529017b84f20c8c8f76e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00245.nii.gz
case_00245	segmentation	241698	f0094a4cd26e30fd3132d6d64008dbb722dd15e63777deb9d89cddebfd83089b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00245.nii.gz
case_00246	image	283908783	02ee75d04e48d53126cffd572ab5da7dd46d8c8b505849ce627a997c0b5d26b6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00246.nii.gz
case_00246	segmentation	850390	4fb7b0340b71c0f7a17dae3679b6a363a1ae94f272f9fb29866d40668ff8f467	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00246.nii.gz
case_00247	image	106016260	ac96964f0308578a0d2e1a2a0145d228376768b00bff7ce27c90fe7814e29023	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00247.nii.gz
case_00247	segmentation	360219	a96df398eeaef80ea082984f0127aa883ab0e1113bd7108e011837e1a7e4e5a6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00247.nii.gz
case_00248	image	26495194	a51ce111c6e660dbc2b3e6c8471310dbd9be635ad4e8729f16931ca6b7f0d810	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00248.nii.gz
case_00248	segmentation	124295	c5a56749a8e33f15c7a8c9841ceb14b3fcdaf9334b0dae6972cf5da932d14761	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00248.nii.gz
case_00249	image	118824802	dc8ae3cd72ceb3f43992838418ac97c973e50101a518fd929a0f40311ff780b1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00249.nii.gz
case_00249	segmentation	399762	9a9a4a18bc9abeed689715bf97083b416a2315027a1818bbb3dda7b22ed2df94	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00249.nii.gz
case_00250	image	97097990	3c67d3baa9b779184fd5df4b29972a1967161f5b308e4e4051760abeecee64d9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00250.nii.gz
case_00250	segmentation	328878	31099b6efe2c4437d1294fc989bf476800f9a72b7fd6acfc3e3a4f59b9732b72	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00250.nii.gz
case_00251	image	22695200	09501ac103f0cecd5067474f0fa0616cf88abe51fe92a4e0afb3807f48c4a9e7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00251.nii.gz
case_00251	segmentation	90501	567c8cae71ae46a2a07e4828f6bcc0e1e78cf2a5dc5ba07add14a80d702dd7a0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00251.nii.gz
case_00252	image	22888963	67ef560931eb6c9fcb973726d7b1dac52002bbae15d1084bd8bb5bb769061bf9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00252.nii.gz
case_00252	segmentation	98599	397686e1e56c5caa030d49b66e5ff0bcf9695b2b1d4c281fc740c36e20f175b9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00252.nii.gz
case_00253	image	220845448	8dd3f72d5cdc2c0614651f7909081bca6503fed6603f9e4785520333f88d831d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00253.nii.gz
case_00253	segmentation	667764	d9b06d4fd2bc1164aff4ad81f1acd68e45684f4b02c37fd7b914ea614855531e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00253.nii.gz
case_00254	image	121347832	c7b7e1c8b462844af4c14ae17c949f192460a4f8fc6766d108753b6d120c0634	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00254.nii.gz
case_00254	segmentation	418186	14323dc0edbea3f3317c5bf3914a4ecdb36ae5935b9c55455f78dbf8e17b08eb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00254.nii.gz
case_00255	image	73537507	cf4fe4bee3da4e14a93d8e6282b0506f6b5cf8e6f86b94873c63f610c624020d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00255.nii.gz
case_00255	segmentation	271987	875263f900e921da4f946d916aa27ec2f644990b28316e615e20fb538beb7db8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00255.nii.gz
case_00256	image	30832286	40759f7f7e58cbb78f8848657174183e3e96a05e92617d37f13bd0003d1dc6f2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00256.nii.gz
case_00256	segmentation	127533	1aea3dbc3155f9a303f133b6f3902b96c1b40d3fea0a946199463ec74d7c5bda	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00256.nii.gz
case_00257	image	15440004	b89ed38bc0575d75d7ceb3d93423d602d034f0c13980f8898d175fb12aeb1ce0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00257.nii.gz
case_00257	segmentation	71384	75099ee56ba58c0f6d8c09829ec9aaf93cfe790ab26969fb4adf55425b79e05b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00257.nii.gz
case_00258	image	44551093	e50bcda46c997d0c9fcc420a46468990c146b230172ffaffbe507e2f24ed53ea	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00258.nii.gz
case_00258	segmentation	183250	c2e1741a3af7b4b082e62c470c81378ce3ae226f18682252f35c1eb53c4a40da	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00258.nii.gz
case_00259	image	38176565	71b545f65551b75b6377367e986314e558216ef9b370bdd244385e9e9cd994a6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00259.nii.gz
case_00259	segmentation	127743	c3859a5a902b4e2affef782fc663891b1e95f55ab234ba744e31122f7385eb48	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00259.nii.gz
case_00260	image	25637986	7043359af4b3f9e43b639a11597a5e2391a9220cc341527b10fc4ccd56a3599a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00260.nii.gz
case_00260	segmentation	93230	e98449371045c434428542e7f1d36f5f7fbbdbdf9085c168c9d261e123f05689	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00260.nii.gz
case_00261	image	255984897	8348924956e269ce03878f8e860882608b459623933152b13d892d9ed31d0b46	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00261.nii.gz
case_00261	segmentation	811959	dbff267ece26791b6e441586d42f172cd9a4ebbac8c0109b314e6fc811b530c6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00261.nii.gz
case_00262	image	31776703	ba94630fbafa61e9af70879fbe7604059e17af738a587bb50036f5b8f6f40828	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00262.nii.gz
case_00262	segmentation	155694	891ce66e24c0491e7b2ea8c3bc2c055fc19225feef9226af9d79dcfe3cdf8fe0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00262.nii.gz
case_00263	image	310741526	e1f2293ae9c8c8a643952fd8eca612e12073677f765375370e832659a3f4f8ac	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00263.nii.gz
case_00263	segmentation	966548	7e544422494218c21c29b91b430e12af49439e014507aa5a877a3334c28847f2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00263.nii.gz
case_00264	image	189605851	770fb9f6acf715e232d0b2aa3c1f0ea6f6edf648bbcb6a2eeff0df4da16df0cf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00264.nii.gz
case_00264	segmentation	568526	0749e646551082dc5f6e59e14b95ec0862a940d4f00d0ab11d87d840b402b114	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00264.nii.gz
case_00265	image	45731190	f2d79583b0d6d88fa8b9b86e8d4358bb137e94cca1e32c10dc30bcf74c267967	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00265.nii.gz
case_00265	segmentation	148140	bb2162f2734950ddac40853fd20e1d86ca97939ee39b2e805142f2d3918f6770	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00265.nii.gz
case_00266	image	42291072	a2eb34b4e4dbd6c60a2b762fd06ecbe9f23cbc4c16b1dbc07ada9a3e5b181ca7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00266.nii.gz
case_00266	segmentation	129374	877ddb8b5cc2be9aaff8e3bddeecce2ba4fda489e39788bef9b9d2d1351a4f16	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00266.nii.gz
case_00267	image	23594969	c50f4bdd59af4ed58773e66f4d53667299a386ac2624587ca83783f30116b9da	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00267.nii.gz
case_00267	segmentation	86654	38c888a27e90dd866527e5980cce3d8b3bf017e0e969e2336019967e3d5efb00	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00267.nii.gz
case_00268	image	80600173	1149453e8ba194cf49083f9fc7d1697d4f6f090914313aaf30836b7e11d73623	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00268.nii.gz
case_00268	segmentation	352314	bab981763953c409233dcb489f61775d9c20ff8b0db1c18e945f285c79e30db2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00268.nii.gz
case_00269	image	45354852	77ca4d9923723dbe58f55630179ddb3c84a22f351ee330eb3565f13a8749f163	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00269.nii.gz
case_00269	segmentation	151410	29c71a7d8c9c9e369ce61b0bbccae186b90e9657f0977de5ec77c7ffafd8149f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00269.nii.gz
case_00270	image	47591806	f2c71c46bfaf186c1d2869dc29b9f6eddf80c0567cea78ee6c777ec5b2b2692c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00270.nii.gz
case_00270	segmentation	166926	aefd25e3492df66c095726fcd90481bdb2f1e4fe71f41cb2ba3b818f242b7e8f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00270.nii.gz
case_00271	image	216019746	e90292b783fd3b24bc81f9fdce1512ae0636e2d5614ae7822316148beff382bd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00271.nii.gz
case_00271	segmentation	761289	d571bde6d7721980da0372e3ab301653385abeb73e0c8864fa4d6747c4693712	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00271.nii.gz
case_00272	image	271071212	05d082d95cbc5b84e366648d0879516eee62e725221d32907008f08fa7958846	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00272.nii.gz
case_00272	segmentation	844261	e78d9716465fa6871243c24ad3ae13d97d62e0b0d47f4c3625d014d4729e1270	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00272.nii.gz
case_00273	image	110309587	63be488c998e496bea1dee403b9a6e57b2d10660b0d2cb83b187c7da77174062	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00273.nii.gz
case_00273	segmentation	360265	e8ffaf6f9562d2abdccfce04843c38f1a1401376c290db536061350a6439c739	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00273.nii.gz
case_00274	image	54524074	51894302c01eb22fe4ca5b211eea33eacb93fa22053aa427de70a71970f6d424	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00274.nii.gz
case_00274	segmentation	209598	d7ea89e65138452944cb1a50dd52eabcb05f8ed4aafcf3f3d4190b6565abd7da	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00274.nii.gz
case_00275	image	38690355	d791a70d571c99dac63d8fa8d8b9ab18e8f2d3a177c4256a738f9abf3854a9ba	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00275.nii.gz
case_00275	segmentation	150173	b72ccc03896b1aa6adebfaeab19885b6395aeb0067beb9cfc48282f25bd18f46	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00275.nii.gz
case_00276	image	305614995	89e90757c87201d0fb21218519a86fff630b5dd68b138e211f66efa14b1138a1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00276.nii.gz
case_00276	segmentation	952770	0cf6c71034601cc03efc59fd54e575c635c11228056b41919692d0c7df400e44	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00276.nii.gz
case_00277	image	40912964	12eb560334f97fe28bdae58f3630b97d30684224297896650f538c654007bbc8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00277.nii.gz
case_00277	segmentation	124453	8e66fc860008fdec28f16df2e16b13207d736225227a930aef95ffbe99207624	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00277.nii.gz
case_00278	image	82590715	ef321c8c209689faad4ae5ba876bdfd20bd0bef0d310dd2605b0932beabc2233	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00278.nii.gz
case_00278	segmentation	309577	b1638c246e8d934000ab44335775fedf08baad48f58a39e2f2afb76f65655fde	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00278.nii.gz
case_00279	image	26124728	f70aaa35e433638b60e6d37b53572c7d7f234dc636ea108ad7cd9d77d47a0816	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00279.nii.gz
case_00279	segmentation	141657	a500a1f6c0423cd95de0136333767227d5ee46a6c3dfc0501d87667450c6c14d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00279.nii.gz
case_00280	image	38398613	ff7c0e2621bbf34c4d6c0ca748842f6207fd98a4996459c4516ba43c4295b2de	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00280.nii.gz
case_00280	segmentation	150892	be38c367be996045c3d9997aea7c7fab5187ed88c9128eec09fd8d2b06cb3728	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00280.nii.gz
case_00281	image	123692886	94fd0509a6992a21aebb232370f7017b2d84dc6122f5d28d42b20378a84e2808	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00281.nii.gz
case_00281	segmentation	413261	365ddcac42ad0437809f81dd113540cf2550f157fa8c70267f2688697d4851c0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00281.nii.gz
case_00282	image	42431662	db6ba04211fb811bd0dc8f060c91a67483471f6ba2b54c7bfe4e651a4ecd5a02	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00282.nii.gz
case_00282	segmentation	139023	4e51346498d5f3381313a1e83836c1788dd1dd87e223ec1f548d4b3d86b7cc55	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00282.nii.gz
case_00283	image	158916095	32bd4479e34fbc7744e5825662c7661bd703cc4809369ecf613b48837c4c4757	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00283.nii.gz
case_00283	segmentation	515033	84158fcb380578b3a080a6db4c0d8211743a490e457f02372aa6f320e97a531c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00283.nii.gz
case_00284	image	113572559	d0a2719ceeae1b272e29efc16c91439e6fd070977a6037e27b0261e25edd2b8c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00284.nii.gz
case_00284	segmentation	362152	8b3479b42b8bad25c170ebfd0d8d49ffbe2d1e6c6df97aaf648adf960201cffe	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00284.nii.gz
case_00285	image	21479732	ca0c7c50b380b63f1c3bbd88bc84710d6c35d6b0069bb0e8cfd9fee43f8e3349	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00285.nii.gz
case_00285	segmentation	83504	29736e9758671b6654453d76d05eab45b4477f0111312879b99a9d0cb827937e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00285.nii.gz
case_00286	image	54689490	cb73b17a6ab839e38bb3e7d3fd9362fceba9c2207667aafb4bfaa1255e2038d7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00286.nii.gz
case_00286	segmentation	185546	130b4a0e3f8a19980cd46586440cad7a5efa1e936538b073c66a6f5abdc8e7a5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00286.nii.gz
case_00287	image	26272612	37ebdac8b7e513fe93839282931e7f29773b0a3e3c29cabadc200ec7d68b241c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00287.nii.gz
case_00287	segmentation	120563	ff2dcf967425962acdc4257f77bf3fc205ee2287a92fb969b498182f4df907c8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00287.nii.gz
case_00288	image	118662012	358727c3fe852ec934fba6b24291f7d71dee880f3a4705542817a37b2c539d07	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00288.nii.gz
case_00288	segmentation	395335	500cf1176c22783a75f30fab30c7bb2b967c9bc7baa467ab3c39a5f3ce7208c9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00288.nii.gz
case_00289	image	43012953	9294502e23b588e2bc85763328fd7b41c8ff708bb1295dee610429747fb595a8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00289.nii.gz
case_00289	segmentation	126895	8bf8a6c4bb5bf8a6d2f8d5addd7925a97f2577f0b6faa87b3dd5965632b8e952	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00289.nii.gz
case_00290	image	276552340	5f835d15704289c6a0f7f966a917c334036ff6374c603b864565e7b3ff67da92	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00290.nii.gz
case_00290	segmentation	897228	d3a43b48e6f394e90354835a5c170cd445caea50fd5ee722bdde2e4cab2dabb4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00290.nii.gz
case_00291	image	63322052	b261a95d330a15652627967aac72cc31e03670225b1cfbc06db8eee1acb09c68	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00291.nii.gz
case_00291	segmentation	191371	a71525ba09ba9c3603c0f759ce4a2cf3dd4b420bfaafc7f4c11f0abda11b410c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00291.nii.gz
case_00292	image	42335172	40d8a3a11b74d7a92bf2439e1145ed91f05deabec260914b2191c37d67d66e53	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00292.nii.gz
case_00292	segmentation	200891	77c50637402a49769395bfb3a8bb23d23d656aa979d8d70387be60a0e1b33e97	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00292.nii.gz
case_00293	image	264217432	9684cd1ff7dc7fcb9672a836ab9b573bcc7fcec9c0c531a16a0a8311a65c8cf2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00293.nii.gz
case_00293	segmentation	837028	6dbc4cb68b2c79ba3314cbbc2344ea9fe64486863e4659ef15e3e0ff1e5084f7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00293.nii.gz
case_00294	image	286098647	2122c66bc91f8befdc1937fa63d0683c35402d5d9068cbe7a6694c046ef00108	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00294.nii.gz
case_00294	segmentation	795552	d5b9ce00de7a5aa202f151ea397fb38e76811482ba7c41e19dcf2998ee1800ce	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00294.nii.gz
case_00295	image	239083362	8faf1bac54356bf4ae73b554cbfa4734ed0e8f819335aa0935d4be6bf7dab520	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00295.nii.gz
case_00295	segmentation	765148	f77110225a5ec7d2b637e170719031e86a259ed9cd730990466a31a85562c4ed	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00295.nii.gz
case_00296	image	40041438	b5a2c8f0fff9d88cd96d64e1d84287e8b21fd07cbb2b389753d328ba1a460aab	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00296.nii.gz
case_00296	segmentation	130385	5760218f65974648b5e2c56099bd54187f1d2db24a760f34364da28c5098fe62	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00296.nii.gz
case_00297	image	42888680	a1c382a99c77963c0a7068a511f82739de82b5a12b0268aa3252af7bdc786a76	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00297.nii.gz
case_00297	segmentation	149697	3243133a6c85f31f25359d5cf122ddb50b384eac18487f9f55a5b2ca591de396	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00297.nii.gz
case_00298	image	282478950	646b77377efe6c7d5ad424eb83e0181141c0378766ed622123863c3a5d19dfbc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00298.nii.gz
case_00298	segmentation	881256	dcc629cca78665c945f55a11fc390d25d777b4aa72d221dd3fbea16fd7ecb35d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00298.nii.gz
case_00299	image	280551640	4e46e9ba66661962f95d627c5b7301702e7bcd7c7e2efe75aa6b97cd270a650c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00299.nii.gz
case_00299	segmentation	806193	4f5af9d4ad627401a86dd3bd1eca6c3d9d7498fd9f8b567bbf4989e7d1357d12	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00299.nii.gz
case_00400	image	56007981	5f2383a928ce29d394c48092a00fb12607614bb89c8ccb83cc75c2797d9e7415	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00400.nii.gz
case_00400	segmentation	163036	78499c3c55fa4aaacb482f1249c9311bf1d68acac5b169b467305f85d48102fe	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00400.nii.gz
case_00401	image	59594517	13a50c91feff051735588229dcb3f65cd3c877bb67e3beaac94f36a3124079cf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00401.nii.gz
case_00401	segmentation	214852	f9faa7f83e7b9b9ace7513ba6c1376384017eb4785d2cfa990a1451dc870e517	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00401.nii.gz
case_00402	image	22685280	ff26d5a32be105eb6d012f0421af12b17821c74301649e492c13e6fe36245ecf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00402.nii.gz
case_00402	segmentation	81988	a8020550527ca21dba4f1376a5811a5a093e9dcea07fad86a475fb7865dfb0a9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00402.nii.gz
case_00403	image	23865358	04a8a01957baabdd7bfc73758e30214c02eb0b23ca2acb945f7a94015a8f4ce0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00403.nii.gz
case_00403	segmentation	97128	badf928d32141aea3f6e2c9bc2443480d255e7ac2a8eb76850da2a214d93aa16	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00403.nii.gz
case_00404	image	55743561	0861eaa63b86fac8375232c822a785d86af9fb9b517973ecd750a2b0823121fb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00404.nii.gz
case_00404	segmentation	201947	c1d31f3b6ca9afc1ef06e4c5ed47e4d56bc2e739eae6f1d642dd5dd0652a939d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00404.nii.gz
case_00405	image	48926107	efad897241cbc6d50a6003b982bd10d7f486516bac106fe40b3a2c24e65ec48e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00405.nii.gz
case_00405	segmentation	187185	3a9ffc22dd9dcfec1627124fbde0c6d04e5a781416cf3135943c83ee160a8b34	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00405.nii.gz
case_00406	image	34297237	a449ac2ed75b3ee7d7e595fcab633e917447fa73c03fc96ba613a1444c174636	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00406.nii.gz
case_00406	segmentation	129044	16082c483ce322e07164cd6288743708a943d209245aa2b4fe29506818644608	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00406.nii.gz
case_00407	image	47744083	48e1ae7c6ed7a95820a3960836de9f895a7fa75464d5e17a06788685442ef1f7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00407.nii.gz
case_00407	segmentation	143041	55992569b3d1d4b8c10acfbb73d74077ca44e8003efc291a91312f7205447a50	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00407.nii.gz
case_00408	image	43324557	e7e6d0fa85c9ef8c98b5416d324c445a4a8fe937739f97865e7e04c231c407f5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00408.nii.gz
case_00408	segmentation	180837	a4c21c897e921554cd5367b449db6b865cb5ffffd473c901b6a6274ee1d8550c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00408.nii.gz
case_00409	image	46737433	31e73618e147a4996fed7b22bde0fe843676bd757a7cf6e545f995e3ae4226df	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00409.nii.gz
case_00409	segmentation	183021	38cb37be66fb299a30769e56c476172811233de50bf4671859c60e3a54d30450	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00409.nii.gz
case_00410	image	35576038	002eddb9dec0be740f0557a352ff5f3b5de4650866c323de42adbe9867129114	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00410.nii.gz
case_00410	segmentation	131812	f09ad262f32c8bc7b826003f9e4dbf607891b24bbc6f8a7db06db9152bf0c55a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00410.nii.gz
case_00411	image	47299594	cb21901856db3015038752b0be9640c3b2391b8197008c296acd85787a20632e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00411.nii.gz
case_00411	segmentation	152670	3b5fbfccf07bae13ad1a811f5d85ec0e9ae233a2d9c737ff3f28c36a4e47ff6e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00411.nii.gz
case_00412	image	199443805	e5bede2c9946802eff2af5f421f57500b7a1c07891a11a3c5c490d98658adfd7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00412.nii.gz
case_00412	segmentation	624469	90ccc828857026e96823f416bc3c8b6c7f0b83fc07e4975c5260e3511909c8b4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00412.nii.gz
case_00413	image	47599089	158b60625db9c9b8a92b6290ebbd203f9ae653a95cce11abddf611c468651782	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00413.nii.gz
case_00413	segmentation	162543	68cd7cd9847e3828cc1a3ba19e1598fefc73fc0fcfa8f4b4a0e8ad86233e4a91	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00413.nii.gz
case_00414	image	46678480	39447bfba996f13d406a079835194f5fa87325a5ffbb2135382fc67b169935fe	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00414.nii.gz
case_00414	segmentation	157434	2371283de0964c5b9b07dc4c3d8ac688539ded3000907949d173ba7707c0a819	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00414.nii.gz
case_00415	image	63475514	7c9848cc70db379e1999dcad8918c114c3b42111b4f9d0417426d2e9b3e101d4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00415.nii.gz
case_00415	segmentation	232771	2702052f7cd4d63c8f58011b6191bd0ace0eae6ef72f305885e3cd5c6f8d4497	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00415.nii.gz
case_00416	image	23313938	756fa3342bf1e24a6159c5b144863d67d8f207d3932ea848108ba484000a4f8a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00416.nii.gz
case_00416	segmentation	103408	674718948594de32d0678ab44dad8a8db86ec3d9c14ca339d0480ea66c8eea09	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00416.nii.gz
case_00417	image	55588478	9af9d3481474e8ff7aee1378200b8f5821c18a1429fae4cc846f78752ce6b4a4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00417.nii.gz
case_00417	segmentation	214276	b5c99c9c0469d84a2ceacd80d5c1c9154d8a9df38f5d1e6604df41a713d491ac	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00417.nii.gz
case_00418	image	21964054	8e0d025ce13e233b48cb3b8477fc584d0f8f60c12823f5985cc17eded5cb7cb9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00418.nii.gz
case_00418	segmentation	120645	dd57b196f1b8b78d445bda942dba5a09086f92a34a041c31bb50b2b9562a31d7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00418.nii.gz
case_00419	image	50455749	6d3de5ebc0be11aff0a00c7a77456d5257a575300733032a392112e2327598e6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00419.nii.gz
case_00419	segmentation	187537	24e55070ca291e845c22a98aae486fd7449fad882c6fe8820ccfd85bdf2df970	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00419.nii.gz
case_00420	image	45309172	d0aa323f2a3ee4549d6d3886ff1a578e5f4f242f44546db259915478bbbbb627	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00420.nii.gz
case_00420	segmentation	154120	c351371d26dfb5f104e1fe12d7b671791d5542f1a80f6e8121058839cb564683	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00420.nii.gz
case_00421	image	52549011	cebb45afd8205dab3f18e937114e7757764efba53236b49e9c85e71d212af813	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00421.nii.gz
case_00421	segmentation	188875	13c1aa97ad30611d0c967c8279be8590dde99905ee2aac3f1b89509ca31f085e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00421.nii.gz
case_00422	image	20304071	61e727017f33c5ff0444e811ca045dc59a9991a0f6b84cec6ca60ec68989679a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00422.nii.gz
case_00422	segmentation	76359	00e3d82a41e09410c6a4d301556a4d982b8f9de594f4f6edb9281b666cebbfec	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00422.nii.gz
case_00423	image	23721267	ecd858073f74298e6714f93eb574233734a72e2b39d44fb75a9fd846508975c9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00423.nii.gz
case_00423	segmentation	80115	3c859e9dbd5eba3476d4a870e3b8ba76c3126d10a349c7a95685071c0aeec80b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00423.nii.gz
case_00424	image	45131376	153bf9d8a3d80d1622e42917ea489a6dc92d037c21f36387cfe8cbcd74671cc2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00424.nii.gz
case_00424	segmentation	138775	ec097d95bfe8ee0f455d66b91b3ad94f1a6aa12b62f8a4f091483fad11e711aa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00424.nii.gz
case_00425	image	26249185	071d63c82682b88d7765584645013d5efb3c23aaec48c545dc881b75accb9ce7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00425.nii.gz
case_00425	segmentation	98919	7b2b32bf41f802d3b86109d59e9abe6d5c2f7f9b9fb44f356968d2d8cc002c2e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00425.nii.gz
case_00426	image	42096511	6c94437d1b512bdd07daac90b3e29b498ccff2c26aad51edf2a9ec2f1227c46d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00426.nii.gz
case_00426	segmentation	167475	098ade190cc97175af6877c18d58f664b69350e482cc3bf42985e4d84bd5f569	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00426.nii.gz
case_00427	image	23797688	37429846968801520a2991ef7458152be43670bd42bf659ca097b34c37d975ad	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00427.nii.gz
case_00427	segmentation	98907	37f8394f5fd208d0c33284087879a5c44fde6772a0a524d8be5870ff45e4336a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00427.nii.gz
case_00428	image	47215023	e94ea76691f4d7ea86fe2053f70f0f19b382d93db59f4ca874e03af5f31b723a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00428.nii.gz
case_00428	segmentation	149743	184c2b923a51bdb029a013b003c9aaf791c04a8aba56ebd4dbef21077a008ff6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00428.nii.gz
case_00429	image	36900765	08e2f2d8e40b45374930ac646f97ee8526a4432188a090b23eb83c2974c53a55	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00429.nii.gz
case_00429	segmentation	135575	210812214970d95b5f63efe444c3f62f800cdb122f2b3d9cbd9c401ea04fdd1e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00429.nii.gz
case_00430	image	27837560	3eef3d2346f55bb980e67c3ffd7e0682ccf1fbb9dc5951ea77c4733ce4157bee	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00430.nii.gz
case_00430	segmentation	178767	fd4ff76bb447021d25d72aeba8d1f193cbf7a55d4add45d0e03084c19d8f41c7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00430.nii.gz
case_00431	image	60283581	e3ac68eb9ac60b46781ffa1beeb9dd097cf6656ff6712f483f641a5ef649a70a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00431.nii.gz
case_00431	segmentation	194671	bd989761cdc8c73786e1b2ef6c641f773bf8cd848bd25c6a7c64625e7d6f179b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00431.nii.gz
case_00432	image	77880617	e895afd08281734391901b42516cb627d680de6f03d05b40839dbe3653718be6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00432.nii.gz
case_00432	segmentation	255041	93b230bdea16dcc6128b5c601921b7c7ae6ab275cf751cc6bc1a288e42a36eed	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00432.nii.gz
case_00433	image	52403076	7ad071e9832876013af8102a5883dd50278da3f0861921ba583b52fee1a6586d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00433.nii.gz
case_00433	segmentation	197512	9b8c73dd8a00e60919247dd2d5b3ba2029d4d8da613de86f48e1d5b96eb85756	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00433.nii.gz
case_00434	image	25239951	6a81edbf68df95c011532ca8cf53d515d5aa4c82557bc06682f58c6f232fde41	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00434.nii.gz
case_00434	segmentation	92565	375c7050719fd75a24b25631b1a74c5a214b59fdec1c8810a3ea9e167bc4b669	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00434.nii.gz
case_00435	image	22215719	518588211eeeb3868acf254a3581b07694807c8520a69cdaade97cba9ed17919	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00435.nii.gz
case_00435	segmentation	88361	5467c8d519c011323e5e96725d837356f48287300389291461ee08edcdebfd80	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00435.nii.gz
case_00436	image	23645233	bdbabc2bc0a3bcfaa8b5313afa9e2c01ea67ae7f6e97ca05a75a4252a5151ac3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00436.nii.gz
case_00436	segmentation	89727	be62700b5c6822049c80b79e94eab39bb571470573587400c374a309fdda99ff	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00436.nii.gz
case_00437	image	64559138	9dbc7fbc55b9d2ed957151f432b8b6d58b684507aaa687cd57cfc2cd4411996b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00437.nii.gz
case_00437	segmentation	219837	c36d3787ecf873bf5454a0ce508a872dc8c7dfc813890ecf1ff5c508059e0a68	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00437.nii.gz
case_00438	image	47828520	444ee0071ae32e4a4528c55847d75bf9251cd4d3dfdc1221c87f3d935562475e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00438.nii.gz
case_00438	segmentation	158641	a4bcfdee05bbb6611cd424a2dbbe8ac83d4e3635b1fc22c7fd799a6acd1a54a0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00438.nii.gz
case_00439	image	35237260	6143c95225215709812519e4c0413a439c7adf141024dd45bf572464da7c5911	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00439.nii.gz
case_00439	segmentation	123387	1d5f23e08c90076f74285e0793749cf501110877e1242a2caa03444dfa93763c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00439.nii.gz
case_00440	image	23141872	25615ad0c0c665936994844fef122baf2467ffdfb455ffea6df9ff4f06faa652	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00440.nii.gz
case_00440	segmentation	84051	32b271944331b90272d644078295ea5abba14f3faa8357ca3d6e330ccb8ff55b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00440.nii.gz
case_00441	image	26304709	d9da3ec24e555ed8b83c7f81b9ecfc385da310def0174bb4f8e6dfe3efae4f81	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00441.nii.gz
case_00441	segmentation	88421	61c014b5b3f6279f95f9265c9f103d12e4cd3b411509cc1a180d392fc7e1fc03	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00441.nii.gz
case_00442	image	392115746	fd74109aec787262988a3112b590d909c3c3ed4dbad4b8f968bece191f489a85	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00442.nii.gz
case_00442	segmentation	1251086	d24734eb20c2374e5f6747557db8febd8cc498304f1104093eaf8713236d461d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00442.nii.gz
case_00443	image	29452614	5cf4df800d4ad995543383eecbe20aadc900cdcc9007594c813b90cb14825e8f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00443.nii.gz
case_00443	segmentation	111825	ea35d866af73ff578fdbe168c1b2993048c49b6e9d4d05b462cfe497e6130b1c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00443.nii.gz
case_00444	image	48752511	f900a151988753a5fe7dafb06b2e743e1d870612c90a7a46ebae66e635525bc7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00444.nii.gz
case_00444	segmentation	154337	990dbd86c968e14331662163456bc998a30c27eb9db0f3e0926823902be5c3f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00444.nii.gz
case_00445	image	49815734	aa29dc8659d6631312cf4fdad133f44355ff0655d7b193d4c27540fbb36ebe17	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00445.nii.gz
case_00445	segmentation	204112	578d9b1d1c8d6196874eae425f7a5a08d83f4b4391f682cc474804de84e4f640	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00445.nii.gz
case_00446	image	41988275	a0711a93f80f90048a6b3ed980cbccf7d6290bdab69d9708be4b820712778de8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00446.nii.gz
case_00446	segmentation	155727	075d6f354b2a4181ccebd32274f58d5759e6c3dc1c9d013f1859cd7605806400	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00446.nii.gz
case_00447	image	77264692	ce66a16fa2754a5c4c90fd044deef05ec2d44ae4f2dcba15b2933fca5593821d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00447.nii.gz
case_00447	segmentation	278824	05af1f81c1d31b4ca9f4dd97c4d17479ed46990a66f7dd92507c55ac9398d7eb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00447.nii.gz
case_00448	image	111748759	32037c53cdc9a51c250f37d69d5e66cf8a1cb6cfef8dbc2be67989e83f476228	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00448.nii.gz
case_00448	segmentation	320931	a528ba8aac1bfcf954e94b2c11eab6d536bd0acd445e03c94273de1faf5ce873	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00448.nii.gz
case_00449	image	62312122	a0ddd4bb48db2aa9515715ffada666675ef2af66975716b841d11a2b5b43e512	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00449.nii.gz
case_00449	segmentation	177399	d9168c36eaeb64d680ef21fc8292e3f1cfe32fb845ce9d6590d079093074c8b1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00449.nii.gz
case_00450	image	46762119	7cb7da1fd88fecc163a3747ce4ed9086b87ea11bba596f094cb8a24d9134cec1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00450.nii.gz
case_00450	segmentation	153009	d6c5499f587528b2c3c91ca269b9809622c9df48da07144cc58522ecfbd43b95	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00450.nii.gz
case_00451	image	61697557	72ba456cbceccb25e807a3c09033692466dc2b0367699c89ea4c3cad008fc4dc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00451.nii.gz
case_00451	segmentation	231362	5c58c609983348a29e96b991acbd37244cd6a7538579285bbec156c33a5e406d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00451.nii.gz
case_00452	image	24039469	a3f62054b6544cc173eac864e16287375f0f923762abb23d4a4fd384c18e1a5c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00452.nii.gz
case_00452	segmentation	84185	02595334c33178c7cb01a9c8f5306557f4e4ccb8cf02194885374760508bdeec	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00452.nii.gz
case_00453	image	59428632	0b0f3307fb5c860b47a617bacb78da2f8bbb68da4d2d70bc5af7fa4b759ce3f5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00453.nii.gz
case_00453	segmentation	236094	e84d720082e15226de6b065ccd5b485098ebc375c9a664d592671345f44c4b2f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00453.nii.gz
case_00454	image	31717152	745623f7e08b874e6c1ce32e19b0fdc96a0298c4eb43b414b0896746108d0bdb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00454.nii.gz
case_00454	segmentation	104313	8bbe1d187535051cf2b04ef6b52cef59db5ff8feab6c455abd419fad33677b91	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00454.nii.gz
case_00455	image	74814048	0f063652ae668c729073c158267c4a98bb6a70cfade39bb060fc52a5a6946ea0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00455.nii.gz
case_00455	segmentation	229443	073a067f250caf2cadbd6589d6131eb22baa2c3636cb2ea8f79f7dda94772b75	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00455.nii.gz
case_00456	image	50313331	56f85125e9d0d98116e9dd0dba3d1a0d891f488466b90b6cb35b236cae1b8024	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00456.nii.gz
case_00456	segmentation	154630	c66ef2096efe64c53711d99fdf4f1e7ae0b6f9da487c827ab362b05824cd5668	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00456.nii.gz
case_00457	image	55625299	f9c750ff0392e635f574161ce0c7c960841e6c590bfdb2d5bb99bf89e68137ca	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00457.nii.gz
case_00457	segmentation	218610	7d5c734b7e1fc1f863f08476d27b2604afcad6cce2a57b283a6b52e84fbd1dad	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00457.nii.gz
case_00458	image	35311869	3ed1aac7734d763c02b33ece74bd8a3bdc1fb190b600af69f74a8c9c3dfe6830	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00458.nii.gz
case_00458	segmentation	127771	d59e6e65524e9248034570f6ba3d48d2173d91705078b42dfb29ee309c4d7ef8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00458.nii.gz
case_00459	image	21928040	df0cc393d849c8a59ab699afeb3c452baa8b79e689fe3779ae74de93d7d0c27b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00459.nii.gz
case_00459	segmentation	76205	9bc4eea64c02cfcdb7774343be9651f057cb9a174391377aca1ec151e567e735	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00459.nii.gz
case_00460	image	37887175	de462c497391f400944ee9716f20c80918aaf850c9c8109aeaf99b14809071a2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00460.nii.gz
case_00460	segmentation	140722	a547350366f0afca4fd9386e086ae03c1ccc01d05bfba7d19d519e3d8f9f84c9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00460.nii.gz
case_00461	image	67667646	00bbcec62ebab02efc7a39f178d295595ec6c8e33fe1fae55d92af4b730f75f2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00461.nii.gz
case_00461	segmentation	212373	9d96832f59cd6d74f24a9100749fd35711f726114fb3a2cdcf3a5357e1d58ad0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00461.nii.gz
case_00462	image	53586787	1b6feec2731ef83e29869ae2c8d77a4474832e746fadeff49beb17ce89a3abb1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00462.nii.gz
case_00462	segmentation	157500	d9e5146747cef7c20fb661a256a3c47670a9575d9460b31a2cac8cd1e965d72a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00462.nii.gz
case_00463	image	28477335	4c0b57858a54226322e925c36bccdf2b2bd30c99093790a65265e6258ba7b759	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00463.nii.gz
case_00463	segmentation	96704	16a7b3396bfd59623ec0e9819b3b3e8d0ca47ddaed18857bc2c0ffb7ed898e1d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00463.nii.gz
case_00464	image	23573684	9e8e90d2f2eedeaac2ffc69717297a10c11908c2e4688aa82e46c4ef331d32af	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00464.nii.gz
case_00464	segmentation	92710	4705f7b1d9e5ffc34ff184dbfbd73258cec7991d7ebcda0f2500140cb2ae76f1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00464.nii.gz
case_00465	image	47354868	474d86bb7554c89e76de2334d852f708369fbc911ecae5622cdee481e180c305	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00465.nii.gz
case_00465	segmentation	205670	f4be870f4e495a798871ac69987d4017ef6694005ae07254bf87aef59b341ed7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00465.nii.gz
case_00466	image	58076850	2611b7866c9bc3b1a6df4573f4c340d578595983cb75f39c4de42bd15396cc4d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00466.nii.gz
case_00466	segmentation	180441	15c01a0455c094101d4ebe92c68c7df47832d276c27136f4968864f20a9d49b0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00466.nii.gz
case_00467	image	40596049	3522e4ef07cd3c3ec5b5066e24d9ef2b10378835dc0d50ed24a1890f55a08f00	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00467.nii.gz
case_00467	segmentation	186742	21a0f3fc55fb632680cde6ee4d8ccf7fa16fcca402148ec45c9ed93b0395c69b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00467.nii.gz
case_00468	image	26481320	edec750dcff365f79e6eb97212c8178ba9b1e3812d58a6c87e42db56f477fdfd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00468.nii.gz
case_00468	segmentation	127024	f71ebd719b132908285fdb98b2abfe68abe4c2b9723f81b7aa91f81b05d37008	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00468.nii.gz
case_00469	image	41426774	e9232cb30363309cd34c4d62df1875a37c4ed20d0ce73005bd58e8550cb64f07	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00469.nii.gz
case_00469	segmentation	130135	dbfd5b1d35a6e01fc5f97d8a8797b7cff18fba96109369980cbee70a6747ec58	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00469.nii.gz
case_00470	image	21736515	a278ba5460220f38cc5f82fcef1d6b11198d61656dcf1b1f1e2a5ba506dbafff	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00470.nii.gz
case_00470	segmentation	95691	6c8afca85c9c5d962c7c5f03f432b0601114d776eabf36f196b31ef931dde57c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00470.nii.gz
case_00471	image	27445620	22b18ef9fb3e24f949647f3fb86ba336a297879bb2f19061dfd433ac9b3f27d7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00471.nii.gz
case_00471	segmentation	104435	a3c428de9fa55156627608a92ea3eff9d719813710a63fda58c82bb83471f64b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00471.nii.gz
case_00472	image	64115565	745084b9698840ddaa29e81af9cc22e74fb873216927715866d6f03a622b06f7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00472.nii.gz
case_00472	segmentation	241122	ee84621eb706117cb011661fca8f98776e3944518cdaf2d8485edb74aa105ef7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00472.nii.gz
case_00473	image	22773885	b007624468a03576b4244508eaed1cfbdb45ab7703ac8d05c4e090f6340cf2f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00473.nii.gz
case_00473	segmentation	88854	378232dd52b7d206c80c69d4f8f37bc2be40982c7017c1cd3eaf96649978f432	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00473.nii.gz
case_00474	image	78664012	db530cbb7ec558e8e8974c529adf41f35db14cf28b9028eca43a2e477913d386	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00474.nii.gz
case_00474	segmentation	298994	a62c6dcfca3d830f1269bb2858c85602da8ebd4417df6725262c171a8d448a6f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00474.nii.gz
case_00475	image	46184916	330811ea729e211317d13ec5048440214abb4c4d762fefd88d70adbf9e14abad	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00475.nii.gz
case_00475	segmentation	178883	31e58f4e507e87155edbba8866daa1847f1b14e3d1909b72c96488ae6367f8cb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00475.nii.gz
case_00476	image	26027896	b9f6245dbecfb1c4ac341352c5806a638094a25447ef79aac6739405dd5fcb2d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00476.nii.gz
case_00476	segmentation	113091	ce7a9ae6fef12923045ef82a56f8e447b986172935ab248ad8d3fa3a522c169a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00476.nii.gz
case_00477	image	37746653	dedd76c3bdf2f74b18508969ebdb715a00a814173636d4f02a2fcfdd55bba45b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00477.nii.gz
case_00477	segmentation	144421	310bd8aa3fd4deb5406c78f16167db64d002c2effe38db8701f14cb8e6377bdf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00477.nii.gz
case_00478	image	47666848	1699a935f56dc99a070bcea1c56edfcdc893f43930d98b29dd3fc0fe0ee71693	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00478.nii.gz
case_00478	segmentation	158531	e3309d925b4294815daaff126886769b97345e9b24a37ca8d1028888952c1c53	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00478.nii.gz
case_00479	image	67025330	8e9b9557550d41040d275a8b09cb2a8abef68d1aec33aa051620c180d2a6ca56	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00479.nii.gz
case_00479	segmentation	274211	c8487c1ba48ee5d7042e9f5670a4b4763955465437a1551e66c0a52ed05f5128	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00479.nii.gz
case_00480	image	55635134	44556437ef2af53891e35c9b1d69f50e9d5ad087ad758ce694df77d405b0d64a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00480.nii.gz
case_00480	segmentation	246605	7cf96c985dbe8b2bf8e9b79aa92ac15d5678ad695791781bc91efee7d788045c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00480.nii.gz
case_00481	image	57384981	34a1d7d548a73570889ab136275ef65c95c178a674ad8fd97ffeefc917c50674	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00481.nii.gz
case_00481	segmentation	267469	ffb72475e1bb27ac75b5dd75126256f2da50012b48986a4cf5bca5038a1b735d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00481.nii.gz
case_00482	image	117731188	56666b4260708c396173ab33d557b4288437a18b8b72a461ac0c3ca267ec0b86	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00482.nii.gz
case_00482	segmentation	410706	75ffa27521ad746f0340e0b20107364f855f2c325e5b68d9116c545a507b42d5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00482.nii.gz
case_00483	image	25643879	26428a17b9d8b67ef1e1e6499a1dc004abe1f888e60a5bc36e377699e0f9cea2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00483.nii.gz
case_00483	segmentation	142596	0281e12d53bc735ffefc8885450ed51af6046b0d07624c158e18d590730920f7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00483.nii.gz
case_00484	image	26724850	9e32b0a5f618d5366dfcac1e217e872a669062a82aec5336f0e95c6ec952ba12	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00484.nii.gz
case_00484	segmentation	97629	fdfdd1fa6785484bcbdc492fd370aeee12fd46366c0c1d53cd427f6406a29311	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00484.nii.gz
case_00485	image	32103136	1294ac6c3b3329b2ac569343335c193594081e8d8d13eabc05538b9ae891138b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00485.nii.gz
case_00485	segmentation	143522	f1e98da6f98468e3285e8bd8bafaaa38adf2ab1ab3f5db2be8790b023057e1e9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00485.nii.gz
case_00486	image	73034232	2f46b58ce83bc9432a0ab8bcd4f84f0e94b6ab23b1ff88bbb2efd6628826113b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00486.nii.gz
case_00486	segmentation	229346	5e3f1789ff2c8509d0d394a14dd28a6aacb56c857d6eda9b77fac5d7d8065c9a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00486.nii.gz
case_00487	image	81106036	2d4fbc323b041bc45ff9bd2fcaf9301ff0ec8cc14b3c76073bb041b36fe1eb3e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00487.nii.gz
case_00487	segmentation	286542	d6a788843c2b1622816bb492e9e458a8bfae487371eaf51c1c14c97943fdbfb9	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00487.nii.gz
case_00488	image	73987897	e91c73accdb754954c64abb2c62dd17ce143f98178709b61b3abba6f0f318e7f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00488.nii.gz
case_00488	segmentation	261728	fe73a1e99eacf5aaa0f81cc10bb8415fe7db5f6393c3ab29a3f9b1bd4e270117	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00488.nii.gz
case_00489	image	72113584	9c38fa4751feba4694047d339c8772bceba75e0a335379aadc866f396a86670e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00489.nii.gz
case_00489	segmentation	261079	031c8b0bcfc34fc8bbff86d5698ac08e62ec4b76ee5d0df31b64d4d26dfc8bba	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00489.nii.gz
case_00490	image	83746361	d4ae1b0a04a1eff4fa0b95e8dc6ddbf7a5ae1508c3a34d5e804bce63f84cc474	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00490.nii.gz
case_00490	segmentation	277850	d4ac00cd06ab6edfb032ac56a92672834e7019cde15248f73ce50077d2373c8f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00490.nii.gz
case_00491	image	24112033	0e24bfa6f94493f606f63ed744d38b7ab34aab282ed4275ce4dc6188e97ebb48	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00491.nii.gz
case_00491	segmentation	109660	4b5cdfd069d58044eb8660bada1f681f72f42872d9c844ad10fefae9cb7a346e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00491.nii.gz
case_00492	image	45237192	29cc81964c5d35aab472577dd05eec4ef156f814bb3b21715eac500c69b4095d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00492.nii.gz
case_00492	segmentation	169089	94cf77a73d6609e28e1855738c53e25860a609bdd8aabf20f418169ff62ba19b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00492.nii.gz
case_00493	image	65782574	762305d7d296a37f963242521a57d271106cb207238cfdc52c1ee8a5b8046dae	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00493.nii.gz
case_00493	segmentation	211904	5b442691ef0dbcf2aa9b4a997bc23bd2494327056964bd4c60505f25b8960446	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00493.nii.gz
case_00494	image	41023184	976f991cef5a733b9798c2bd44398f63e84e11cc4673e6419b3bbbb8ca9a67a0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00494.nii.gz
case_00494	segmentation	156662	512d78eeb011a15869d8789ec3c8c5cf4319e500f7f5e0f0ac5e093754256593	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00494.nii.gz
case_00495	image	27072731	1a59dda336a322b1076badc03742fee71e3c0caaaffce195891b81f11fd385ee	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00495.nii.gz
case_00495	segmentation	109375	15b48e438866b7726d86e7db2154deb47c61f9f27c8cfd6cd4596772bf12d346	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00495.nii.gz
case_00496	image	27071494	db7bd841a6e5fcc80e2213fee90d456acb5fcfaa71f858b7fdb0e3e0b3f96331	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00496.nii.gz
case_00496	segmentation	101330	c9ff71dcb78eaee01cff3d7f5db22848cd2ae52d3c255a0bc9c13a413c52c327	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00496.nii.gz
case_00497	image	18276496	dbecac20d81ce9fe628dce1983418cac926642276a55932a68d6cd5630f6dd8e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00497.nii.gz
case_00497	segmentation	76568	c0d26ebfec1cd1ce536624ce9d563c14bcc0c74acde73b9ec76fbfaeb6b5d087	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00497.nii.gz
case_00498	image	40928883	fec0c7e1faaf31165c88f757b91d3525ef0f7b47b7ee0a5c2e2b6d10805acdc2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00498.nii.gz
case_00498	segmentation	137897	603711f71d785afd940d5524a5594d600b6616c7bb6125d4ea39856936cbddfe	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00498.nii.gz
case_00499	image	27112186	12976f684a045ec5fd80d5e594aac1556a0df687b2759b856921bb9882ff3253	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00499.nii.gz
case_00499	segmentation	132842	662a876e5b8083a38f4e3a2bffb65c6e4854fe7b5d31954b5c401af37209ecc5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00499.nii.gz
case_00500	image	61144340	f46c5d4f7ee40ab50dd961fda5cadd45f41171e467db4df6374bfc1b24721968	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00500.nii.gz
case_00500	segmentation	238952	495cc6cd1f7c44895846be0c8c9899c04c04a34bbdfcb07102a1229e7b69224a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00500.nii.gz
case_00501	image	256478579	fb8d5495ddd11737ec34963932f609ed40aa17857885746d1aa4200331e83506	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00501.nii.gz
case_00501	segmentation	843338	c5835513b1d9dcdb1561d8bd2c97814e663ecb5eee7835168f4a138fb55b653a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00501.nii.gz
case_00502	image	67227956	edcdba04d3d384c9f0e5786594c0f4d122098198839249dfe44c48a33638ac6c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00502.nii.gz
case_00502	segmentation	261051	d62e25c9b0f54862185e13a5c8f83c347aebb132fc3e4081b2e945cb17d79068	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00502.nii.gz
case_00503	image	43922751	17215f0d7c200a45fe65cc2615429211b90c714ba4811bda0b9c81e302d0d7de	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00503.nii.gz
case_00503	segmentation	157901	e66d309ffcef9c45b36cb38698541817db96ff3792780f26808b1ba5b9e0cd07	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00503.nii.gz
case_00504	image	93451921	2fd5c100a2a79943a534159d61edf598b4d4c30018f6bcfe57c492cd58404834	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00504.nii.gz
case_00504	segmentation	349879	404c2275948ad42a68bc851109b3bc2fc4ccec2c0d5cfd690cab6a90ccc84ef2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00504.nii.gz
case_00505	image	114453168	39269725b76afa8057d4c0e2c0630c8b18419b655af0c061b1fd691ebad7fab6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00505.nii.gz
case_00505	segmentation	408105	fcd8dd26ed4341c7a9333e6e5c7a60fb7f268bc10bdf6d56cc8f8b5fdafaa228	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00505.nii.gz
case_00506	image	20394264	439376574bfef70d7cb2f3e4a2d1bc9b227e6f7341223382bec8d9557654006d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00506.nii.gz
case_00506	segmentation	81685	a6f7688cbe5e2b8b58183359ccd3d92f88c324a0092ee6ed1262eb287c99f64f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00506.nii.gz
case_00507	image	42302505	91419b870fe6d37634b9b2a4b0dd3386349d7d5e0ce6fe22d86f442d042603e7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00507.nii.gz
case_00507	segmentation	138712	2236d8abe06586e6985413ac4cbda80f86082982269cd0e01b9c74d996f4e0bb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00507.nii.gz
case_00508	image	40806955	4c99efdb20eb198f8523200019ca1d0b7bcf888afbaaeb9e6e7bc678d63e1e21	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00508.nii.gz
case_00508	segmentation	132850	2de01a3eb668c7248744915480487bd324c0338073ea8bd2d4b7c1f9ae946c84	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00508.nii.gz
case_00509	image	72048822	837cac1570351a80dedf0a69492c27a02420aea85443206029ca9dd5a07f016a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00509.nii.gz
case_00509	segmentation	286109	960d93476200aed8aef9eaf35344204b84ed57bb5cd8022cda22256671e2ad6d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00509.nii.gz
case_00510	image	351483025	1a58f845f25fbd704231e8c34fbed8b36fb5273c14e342007ba2542c0735e26e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00510.nii.gz
case_00510	segmentation	1001949	6caadd645549f6df90e9bba31fc28993f015da712d06b0ab4ec60a075569faa4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00510.nii.gz
case_00511	image	41103909	b7860157bb0638439cc946f26eed7de067093e3b5aab804d15a4e7d7f7e37cc6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00511.nii.gz
case_00511	segmentation	147800	ec574f6fc826990fe33c26b96b6875a46a7436b1cc694a3cea4a48a35237ca17	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00511.nii.gz
case_00512	image	28256326	aad1134851fb68081cf82b2ed583213f3c093fbeaa7926fa6ac745b76874bcb2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00512.nii.gz
case_00512	segmentation	117884	298c4e57e67e57fa9f6a9932fa4ceaf1d751118a0721ebaaa9ef0d9d1c1d58ea	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00512.nii.gz
case_00513	image	205837284	dd6fbebfaef81bcdd2ec26ee820890e0b0885dc4558d318c1342591e22a7191d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00513.nii.gz
case_00513	segmentation	631142	35de48837a2fb8b9ed5ebda99d1221e758fa73df17906f2c332722fa155674aa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00513.nii.gz
case_00514	image	285257667	24458dff74e83b47a1376b041d7c4ef5aaff3fe77a4e1539d19d1c5db39e37f6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00514.nii.gz
case_00514	segmentation	854242	8281cf44045d5703dfb906887752f680c0bf5c829a73541fecb2583292b426d7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00514.nii.gz
case_00515	image	136058384	4cd323fe940b6b4027156c1bc04d8ec7de163422a13ecdbd728376db347debd3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00515.nii.gz
case_00515	segmentation	508444	22dc319c2e3119a59018a324b075bc801e74ed9f00b43ead875ba9d5941716f3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00515.nii.gz
case_00516	image	49760079	ec558682db0b6b5d333f7dcfd8445367db88bc02e90c5645ac4d622a9c409387	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00516.nii.gz
case_00516	segmentation	397888	d96664b3988c371dfecc99c0ac877aa7fd74eef2a9aec6fa4a0130cbf594a0d5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00516.nii.gz
case_00517	image	227996140	9d7bd9b39b240d8e57bca19e332fe41d66aba44b8b503ab03dd9435149c6c22a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00517.nii.gz
case_00517	segmentation	695051	41a5ec9ffe43e3ddbe4e521ebb6c5f7252c377454f8c9683c0d704e20197561c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00517.nii.gz
case_00518	image	67370957	d6fc7ff2790b844ca57e3183aee8caba0df1beb90f2f77aaa4f5e5197c1ae10e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00518.nii.gz
case_00518	segmentation	214362	223dd551ffc9b9af659e9e0230fc91e7e1df0ac816debe6accd8b1ba9a0c48a2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00518.nii.gz
case_00519	image	30920401	2742c108270528bda67d357a5ea343d4503785dc8d56c56b130fc0788953eeb7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00519.nii.gz
case_00519	segmentation	159154	3e4fbbcb903d7cea37dc46eea61babb37c27344cb5914343e0c57691b19c4b66	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00519.nii.gz
case_00520	image	40706659	bdded0c66d217b927c64e4bafab1966c5c6bf939fc0bfe08357241d7f357cdc1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00520.nii.gz
case_00520	segmentation	154025	8f2fccfdc42b17f9519333d8f66e4946005ce0e8597bc9c10d516f35ba62c02e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00520.nii.gz
case_00521	image	40178648	e3ccd334a04df21917a60d1a7d45bd30016293f4e15d28f318ac0a9f5207e218	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00521.nii.gz
case_00521	segmentation	127888	e82e66edbb0e26790babe4fe0b0016661e225e03753f5cf513cb6bcfcf2a7871	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00521.nii.gz
case_00522	image	45029513	0bcfb6d28488c5fca6b6df70dce49cdd0e4e2f752bb5710e6e83a845a07521df	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00522.nii.gz
case_00522	segmentation	147446	42e8281d63bc502daf1a6446a623791951692174011fd3736d487d1216a88614	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00522.nii.gz
case_00523	image	45299539	1f41c819589bbd47f3abebd9c905a935b190d91474d1dc8f3ed2150f9ebff08c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00523.nii.gz
case_00523	segmentation	162743	20b99e47e910fd96e28c4661db53a373574ff9a03b74957d3232d6bae2038755	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00523.nii.gz
case_00524	image	46851687	6ca1afd9b8caff21a07c87a3244661fa7ce3a258df22c5152185c67d005813a0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00524.nii.gz
case_00524	segmentation	155145	287ee5e927faf66adf55a999ca7ff9fa9b5b9c3066b53cdca43883096f46ade0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00524.nii.gz
case_00525	image	41213047	1c3995278a4d441bc7ba29ffd8613d545c84173ff8c93a830ababf289a9910fb	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00525.nii.gz
case_00525	segmentation	186422	c9359206d2af1fb13903cfc36b1d873831b8b0ef60a25bd548f743ab923ebf5b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00525.nii.gz
case_00526	image	67189662	aa4420a8031b0514db660105e262055e287beda1037e600556d916084c476bd8	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00526.nii.gz
case_00526	segmentation	300773	b4182648fb6e0c873344895e11eec0170eec2edccf80c5f387496ed8a1e881f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00526.nii.gz
case_00527	image	14969343	1ab449578015888b9e2b3f20249f42cc339b5132aae473e45f05370c0b34d66d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00527.nii.gz
case_00527	segmentation	85289	c30b8cc4143b620805a544b0e9d0cd2315249076b6b6f215d73dc891a5691179	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00527.nii.gz
case_00528	image	198779066	b7f1d5a03ab706a2c8647a03e4ccf11864285caaedf9080528ad26ff991e826b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00528.nii.gz
case_00528	segmentation	664054	74b8ae930bbffb87cdc1468e64a5208eb906925466e7519b89a6236655592fe4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00528.nii.gz
case_00529	image	21254010	223942b7ce7e5bf962d9f98e4aeb598cd62f92d38fe2514a8c3566512f2cd753	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00529.nii.gz
case_00529	segmentation	82836	f8a98b0d022ed7f52f5979728e1d9a9c236b95f761b8e4fbb7c71380ad4ea39c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00529.nii.gz
case_00530	image	73489126	ccafa1711dfb9895591132a8bce256255ef1b816ee5a4699f80f4ac167366fc0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00530.nii.gz
case_00530	segmentation	267762	fa14261d0c6f27b2325cb763c090dae24482f5c6cd07ed20eeee5807dc56a494	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00530.nii.gz
case_00531	image	28475472	9473affb974d4549ff3e452b5d3918066433fa7551e8470588989e4d65987591	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00531.nii.gz
case_00531	segmentation	193317	0a19383c9db39da76d5d4a70c433699f2bb3ad66295479950655a996fefa117a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00531.nii.gz
case_00532	image	62414728	b86e2b607215dfd7535003b18e1d61379aa991af68d4413ad93ecf44455bd225	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00532.nii.gz
case_00532	segmentation	241591	468b728d6e433d9b0581080e52705be4252b82dd8d55ebedfff11b1141b7a99a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00532.nii.gz
case_00533	image	49557683	4cd4cccae3ea422124c95ef1ebcc36bbc50e6e34f24b88f44d1c4e0377b7c4f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00533.nii.gz
case_00533	segmentation	190776	d48a51a697a51bcb7e2d86b40e7cea0bcf3e97daef8dad829b76ac16b8de50cc	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00533.nii.gz
case_00534	image	16916994	8c8d8618d98830defbf42b70be1c83a8a70046f7835ab31292a9c6e0bb6977d3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00534.nii.gz
case_00534	segmentation	66522	3c73401fd3d7e641fce46b9b5689420f6c819004602e3e4aada1e9436331a95f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00534.nii.gz
case_00535	image	264482826	3fd0d2fb3e2d5cc6100b7eec84230536590a2bbaf549ec704fb5f7ae8b225e90	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00535.nii.gz
case_00535	segmentation	819497	b36acbdf116e3e158a7dcde77c65e9abc1e2c62097a0aa55d318ccffafdb1705	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00535.nii.gz
case_00536	image	43453239	a9445a98b10adcd7f36acbfff2cb0895a30ac6fc7c8cf37d3e3657ae176dd9e1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00536.nii.gz
case_00536	segmentation	149560	f101f62dc2170c1ce29bd357c9572935f978fbc7e41b19d7b5de08dae983e1b0	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00536.nii.gz
case_00537	image	25005535	7acc4e25766e1ca3dfc570bd2be70cb2e37e1c12e912fa3e62145d5c12d1024c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00537.nii.gz
case_00537	segmentation	121849	a918257391dd8e01e6cb41755da46b2e4429919ebe9f7edb880115be9f5898ab	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00537.nii.gz
case_00538	image	173907801	3af0e1566d48408fd6937c6f13413f8a386aea013357b9d995a112d29d68f228	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00538.nii.gz
case_00538	segmentation	576744	05d53f7c0646de22acf9307d98b1e9e96cddc9ca7f973a009874297cdf3fba10	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00538.nii.gz
case_00539	image	17709521	31e85d3bb5645dba56fc08d8a12a7923b424be22c54f83413018216585d1ea2d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00539.nii.gz
case_00539	segmentation	102199	1d6d9dd74f4bcfa184880c8405a9e3dc33c015f804b9f43677d48fc86d7ce0b6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00539.nii.gz
case_00540	image	209563087	7ffd7e0d564363b0e6f6397dfd433159fb7cfb03b723176f6645636a1a569c8e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00540.nii.gz
case_00540	segmentation	608962	5ae57cdf864b204c7d2363f760ae4a8bb4fd9f4e62772a81fd9fc2a7f83da237	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00540.nii.gz
case_00541	image	195829427	e2fc1dbd0c7156b40a0c145478806f16b09bc0c638e957bf8c62169fbfcf30b1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00541.nii.gz
case_00541	segmentation	666237	44e4e4a4560ffec7dfa10512d9ee207894ea21b0d90bed82f7882fcca3243be2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00541.nii.gz
case_00542	image	95359048	771afa981e356ecc37f6e072855886fe3033b30842a93bef9da0f1ad5f3ea394	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00542.nii.gz
case_00542	segmentation	397184	2b897c587b962be60e4219485d2338c99e981bd7959753750673b5950e3efbba	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00542.nii.gz
case_00543	image	215196132	21ce6f1eb8e508f039e140245671976c653b0836fabe2e33afe5e337f34b7d5e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00543.nii.gz
case_00543	segmentation	779879	6b5d675f24f560f20617179caac88ac5761511556c64dc6dbcd36ed28888e9d3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00543.nii.gz
case_00544	image	222874979	d0cf0b63031f6899f7ab0902704c820067df3624d7f0f18e311d8bab77dc435e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00544.nii.gz
case_00544	segmentation	710751	ba97bea92a44e35ed3bee0977e33a2f889ab2328177c10ea978d80f829b0431a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00544.nii.gz
case_00545	image	145320004	85c04da7cfce8bcd9da58c0fd21e4e9c085cd6fee9e478d0aef15f027ae00ad2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00545.nii.gz
case_00545	segmentation	553709	bb7220cfc21fde242fa5d0ba6c38a8a7153d422a493fee8bf739c694705f1790	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00545.nii.gz
case_00546	image	11366210	93250afa24b8dd76decc893d2a4c984ad4155bad7b81aec786a4d52c9ccd1678	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00546.nii.gz
case_00546	segmentation	70701	3420c751a97a3f53487cf183acf44fe89326f4cc2bfd831f1b2d8b13b8b3f1bd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00546.nii.gz
case_00547	image	75081070	8a6a00c0ef356ed3dfeeb0567a828121c75190a39811fb3f20452598733d81b4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00547.nii.gz
case_00547	segmentation	249401	7747fd14425261a139912b3a231860ee8dccbba340b45ccb98bcd3564171af16	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00547.nii.gz
case_00548	image	20897319	0ce4c64c2d65f88183883fa783ad931c1f823d9a903141a5486698184b811dfa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00548.nii.gz
case_00548	segmentation	88637	3a1c896d44256f3eb8bf6d5ce97b23e1601a5d461730858b8287daf2515f8ea6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00548.nii.gz
case_00549	image	23397931	e6537f64a53b6d7bbf410ef07b10d13532e3914c81db2ed7586e2e99a19e4670	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00549.nii.gz
case_00549	segmentation	86799	da44d8a818a82752dcb8482f29bf6adb0d6e4cea7a33865dec26a136e9330bcd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00549.nii.gz
case_00550	image	19078724	c91d31cce7c6615f7f4ed9a63ed35cedb3d0cc0d8fed170264e401949ff434ba	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00550.nii.gz
case_00550	segmentation	73205	4c188655b157147528c51341974d9ce59bcb4d34c15553debf15a506fe49765b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00550.nii.gz
case_00551	image	22290948	ea96c45be3fed1fcf5339ceb410f57bda3e0f769a547232c05c510ac09b40f9f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00551.nii.gz
case_00551	segmentation	121086	98ac3e85c19f768b6c7849c2a9bec4d8f47b45d1a32960d46cf52bdbdbbacf93	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00551.nii.gz
case_00552	image	66938612	38c8eba9390cfebfd7b5b60b6ba3fdfb48965a6f323176ea488935e528a3c724	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00552.nii.gz
case_00552	segmentation	281537	18f1f9023b5c1260408b2939ae9c060ffceddbed20defb1fa60ef46875f7a2bf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00552.nii.gz
case_00553	image	223210983	88df629089665e3e6bb33746b844f14b7b05e3f559a04917eeb3754cfdc33e9c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00553.nii.gz
case_00553	segmentation	697465	4055440c6719c451c6b26fde4d8386fa92250e96a902ff215b7356cff615cc65	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00553.nii.gz
case_00554	image	31450473	e2cdd4566579a767254a35c8add41e4fe03dbd9ea3c89e3d54a7333a541f86dd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00554.nii.gz
case_00554	segmentation	113709	10fd29c788b3d80b4d81e18c05be1c6737ebab20c6f4781239a3fe13c77cac0e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00554.nii.gz
case_00555	image	197494329	80332a4c81f65943dcbc5eb75de20669aec1278444ce85139ef19509f6653840	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00555.nii.gz
case_00555	segmentation	583648	e674707032a484c0a44918e19b85b3df000f51a99d42a120acb8d07281a62958	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00555.nii.gz
case_00556	image	18711613	a124066bc552d2371bc1b35a19b723c8e2c147e38fb396639a70ff8de583f805	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00556.nii.gz
case_00556	segmentation	132086	051502bcf6db09aa1997e4a4431a96af9dd73b5d1a4e923e9195d03bb1901bee	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00556.nii.gz
case_00557	image	78046883	06eb2325c9c2d3b6717a8e0b6bc8058172578ff9c07a26c4b8ba64f577516036	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00557.nii.gz
case_00557	segmentation	274900	c005a41c928b7d76ac42a70ad9ad801d5b1a3f517684b5c502fb088519bf7986	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00557.nii.gz
case_00558	image	62584049	8e3c94f0a820cbe2be0fbdb3bc4465c0a13b62f3981a9ed712679aa9c444f03e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00558.nii.gz
case_00558	segmentation	259172	780e15f5ce3fd96415c210eee8c2e7756f3861f5e4787092ee98a9021df80d75	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00558.nii.gz
case_00559	image	219979091	fc9e1c70bc00212ec3550f623e1920f02e3d4aebe4800dc39d92e2907d62994d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00559.nii.gz
case_00559	segmentation	715064	cde75d1e7bf87ebb57a92513dcb0c9d7a29d7db3fc18f303713aff9a052c5c96	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00559.nii.gz
case_00560	image	51595618	1d891b3e20c589bcd797f9a11323415cdec2476abb375c1c7289212344b43c5d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00560.nii.gz
case_00560	segmentation	194007	8d356e2c41e4818baf9de634683f8de2b5c2a6eeae0817cde7ba6fa8ab27c44b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00560.nii.gz
case_00561	image	238348598	0b3d0c6035118808ae4ae3cfd411aee340adc45467dc64c503afa1028b7bdcc5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00561.nii.gz
case_00561	segmentation	708790	87b3728a58dff57b4fd815f25b9a039023fbe3352b98d4d5e05c13d5f4406ab1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00561.nii.gz
case_00562	image	222585570	796f80a89c3fc7121dcdeedcf81ff5cc4213bf74a6438b598655a43e37ef75c3	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00562.nii.gz
case_00562	segmentation	649361	e4fbed734ccf6fc0624e974115e56232f92d00c83db10cbd15594061448f4a8b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00562.nii.gz
case_00563	image	106746775	17bded08c50bece865608ed35a9ac146c3b3f4e1da2b12cd6bcbf2cb6bea724f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00563.nii.gz
case_00563	segmentation	305414	1e0d92b913b892615fee35744e6c874123ad8548a5be4cbde0817d077eaafe2d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00563.nii.gz
case_00564	image	231755633	70b3361a9f3b2344341326110e4cd3216b7d2b773db224e7fd0aa1668048caa4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00564.nii.gz
case_00564	segmentation	668863	9d8cc2e9bf1fd550fe253d1e8a4bce3e4a78bf19f642a883edf2620a045f9d20	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00564.nii.gz
case_00565	image	22289769	5f809b3cda5e9c9d7916ccba4745e41c16410075e733450a15945ccb72559aff	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00565.nii.gz
case_00565	segmentation	119236	b751be0012940f8ecd761e1b0806d5dbd9c1c05f60d0563cf9f22bf542d2768b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00565.nii.gz
case_00566	image	59738567	ae4dfde8f243e62a303d9b0207a4769982c3a1b881ecca0551758f1c60e92966	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00566.nii.gz
case_00566	segmentation	226382	8988dd6f346d11472e390e4128f5b70d0520946d3df5f985db72fd034dcc3aec	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00566.nii.gz
case_00567	image	78969853	78786557e1d193ed25d62886b4472a9b4ce531c6b64f0867751d33af40f9570f	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00567.nii.gz
case_00567	segmentation	306505	0db70e576a7e44d97886f281d0a18fd0a102674614dd25c3324113bdf1c51ef5	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00567.nii.gz
case_00568	image	76990727	362442f6d600d81bde77caaacf429a46b8d4e4d716e04c4879151d98168e7613	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00568.nii.gz
case_00568	segmentation	308746	bfd981b34aa574f2615729baf83c3e8f4f2893c1ed38ce90cfbf6e14ab645326	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00568.nii.gz
case_00569	image	15047860	66a4aaa552414c8d0eba57a690f0508147867ac65e8e5f51f7d8e70c743ca2e1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00569.nii.gz
case_00569	segmentation	75699	5aa4902e05bbbdc358c1b571e8294651ff077fa5134aa1453939475698e64649	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00569.nii.gz
case_00570	image	21944262	d921ec4cb7908a9de58c6a3368fe069179e4b6ed43188b7b7f18853f8ec95490	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00570.nii.gz
case_00570	segmentation	84241	e8fed3caf00897feef48f9827ebe65ea1631dcd9599ee8615af46e9ff7dc82f4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00570.nii.gz
case_00571	image	33283358	ceedefe5d2b6f37e86865dfadc1b4746be3410600d96e7e8b2767fb61899233b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00571.nii.gz
case_00571	segmentation	121566	0cd52429727ee6978e8468d551fc98dc08d055de1099daf2d6b8c5110accb82d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00571.nii.gz
case_00572	image	215486826	06b8aed1ab889311c805d7a55a4b7b71e7b60737e234ff428d922785b6cc0785	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00572.nii.gz
case_00572	segmentation	880482	461f4afc6948dc997198e1e473d1e74ef78eab377f77664a8d1e8188ac724490	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00572.nii.gz
case_00573	image	39993775	b2aaee126deafce9925b05d42bd5343d1aa102fdb42baaa0f7439048c7348b1c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00573.nii.gz
case_00573	segmentation	152856	e39d1185c9d8214f2f7dba8d725972d4292a08a080cbf6c8d4eba7734a590667	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00573.nii.gz
case_00574	image	26739544	824f0a93f50f9211b96801c350780d084e07d12e78974f562667ff21f4cb8a00	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00574.nii.gz
case_00574	segmentation	101122	1d0f925b66be20816a7d7c520b7bda37488365933f4b719f912ea3d23f375d56	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00574.nii.gz
case_00575	image	100165823	b044af95c7347e564237ae33d0bfb9d0e4a1030d038103eb141629254cc1220e	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00575.nii.gz
case_00575	segmentation	357784	9a701fbb92e9c58d2b8018384ce832aa7cba0d5e125ed3e54e2f2f0d19c049d2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00575.nii.gz
case_00576	image	41439379	adaa8b51fdbb98d758e3bb71a8925a15a0255f098f943bd424f624f762f58183	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00576.nii.gz
case_00576	segmentation	151973	04ad3e6975b2aa375bf2e5e8b0fe6661a0df961d44ae75056c5cf99e230ab6e1	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00576.nii.gz
case_00577	image	54596768	98c805f50ab1eabbadad216f753c7618376d73935984973fd5183e154e9c67d2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00577.nii.gz
case_00577	segmentation	227261	dc4196f0938f029328d8e7ec6c71dea2674763ff4d03127cc4ded7c517804606	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00577.nii.gz
case_00578	image	212390755	9350e2b8a00df6d73ac1ad698c27ee63def7173cadb06707ca2e95fd8c7bfea2	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00578.nii.gz
case_00578	segmentation	630383	4c7c7312a4a48c4c3d58e9dff739848f326413eb8b63d3c002ad8bb1c56f4bb4	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00578.nii.gz
case_00579	image	75798865	ac66b8b350812f28f7de1a9e368e016d21547a166396d09863a566c9d0d826c7	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00579.nii.gz
case_00579	segmentation	278083	d691ef2da3f9aa323417c743b21f9f9e7149704f9b73945cc072d050320da2ec	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00579.nii.gz
case_00580	image	89292622	21778a1bc9c0668fb1b978719dac4c5ccff30b9b547984175a94eff39e67a49a	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00580.nii.gz
case_00580	segmentation	293666	6c7457e8ec113e5684c3cc3c5d80f81d872be5220d0add9c6d1f698c0705e2aa	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00580.nii.gz
case_00581	image	24720462	7051f295fe592e6ccf73bedbe8d657fa2b9282b4ecfb80718e22c93656694ca6	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00581.nii.gz
case_00581	segmentation	90572	882fc72214706aeda7aeb7f4cc6ed03e01d364419fac882f6b74dce785bd0720	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00581.nii.gz
case_00582	image	80200037	c78ffe1ba0722e2901771efee8427c233357fdedbf88b04a280eb24fb5830d1c	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00582.nii.gz
case_00582	segmentation	291413	a85ad115e6fb3877396ee85a8a0ee47829087871ac15389537317c1d13eb9d80	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00582.nii.gz
case_00583	image	122665255	c98fb876906f6ed4240012cda188cf82cfcb2cd7101af33d9f9cee62c762e106	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00583.nii.gz
case_00583	segmentation	394455	71f0d0f79d3da7adb42b6a328879c41bce045d6e04bb5c5c44b79a7889d53973	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00583.nii.gz
case_00584	image	39940628	943b239d06c547cfdc6195ebb7fe7149ed0c8b1f1efab1d72ee2b99b2342708d	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00584.nii.gz
case_00584	segmentation	161749	33d04b7d1153184deddbb654a0248c3d3b7ea16f0f98fbc979dc91c415ccd88b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00584.nii.gz
case_00585	image	73737793	8ec6bd33d517514407df3b94991725f59534b15d696d75046d74d1c1d4bbd307	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00585.nii.gz
case_00585	segmentation	269877	062e691ea40b288ce224c93551ed71accf6e68971ba4c96032e6032e6a27c349	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00585.nii.gz
case_00586	image	72204158	5a0817b9d8211c473c9ca94ef9e5f2aacb55148fd88af0ac75d9ac423ebf0fdd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00586.nii.gz
case_00586	segmentation	263712	ea93e8fdfe6bf795629add03bea0469e4ed36ecbc493a6baeeb7ceebd8513658	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00586.nii.gz
case_00587	image	19855098	825d36f39ff3ad1d3f5a5279138c0a111d51227924eb0e8db27da2f1843ff5cf	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00587.nii.gz
case_00587	segmentation	86523	04cb57e5c0d148f894b7faed1be5afd0e0b5e6a438761fc68d085bc1134da833	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00587.nii.gz
case_00588	image	116233856	487e766c573f3455e23016e3e16952dd1e704da06bd433ac93dcdb36b4e2bd6b	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/images/case_00588.nii.gz
case_00588	segmentation	376319	93768471fcae88350fa89ed47d3d1d7ee230e92ef9c7584cd7920b6536f2c0dd	https://huggingface.co/datasets/neheller/KiTS-Challenge-Imaging/resolve/65f1f295873a326230153c7e1de0c7dba10f0b29/segmentations/case_00588.nii.gz


In [ ]:
SHARD = 2
W = "/kaggle/working/project23d"
subprocess.run([sys.executable, f"{W}/scripts/data/stage_kits23.py",
                "--manifest", f"{W}/data/sources/kits23_v0.1.4_files.tsv",
                "--shard", str(SHARD), "--n-shards", "3",
                "--out-dir", "/kaggle/working/kits23"], check=True)